<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/1_1_Simple_Linear_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part I. Continuous Response. Simple Linear Regression




# Глава 2. Простая линейная регрессия (часть I)

## 1. Введение и интуиция

Представьте, что вы начинающий риелтор. У вас есть список недавно проданных квартир, и по каждому объекту известны всего две цифры: площадь (в квадратных метрах) и цена продажи. Интуитивно ясно: чем больше квартира, тем выше цена. Однако две квартиры с одинаковой площадью могут стоить по‑разному — сказываются этаж, состояние ремонта, вид из окна, близость к метро и даже настроение покупателя в день сделки. Связь между площадью и ценой существует, но она не строгая, а статистическая: площадь *в среднем* влияет на цену, но есть и случайные отклонения.

**Простая линейная регрессия** — это математический инструмент, который позволяет провести через облако точек наилучшую прямую. Слово «простая» означает, что мы используем ровно один признак (предиктор), а «линейная» — что зависимость моделируется в виде прямой линии. Модель записывается так:

$$
y = w_0 + w_1 x + \varepsilon. \tag{1.1}
$$

Разберём каждый символ:
- $x$ — **независимая переменная** (признак, предиктор, регрессор). В нашем примере это площадь квартиры.
- $y$ — **зависимая переменная** (отклик, целевая переменная). То, что мы хотим предсказать — цена квартиры.
- $w_0$ — **свободный член** (intercept, сдвиг). Геометрически это точка пересечения прямой с осью $y$ при $x=0$.
- $w_1$ — **угловой коэффициент** (slope, наклон). Показывает, на сколько единиц в среднем изменится $y$ при увеличении $x$ на одну единицу. Именно $w_1$ количественно выражает «силу влияния» площади на цену.
- $\varepsilon$ — **случайная ошибка** (шум). Эта величина вбирает в себя влияние всех остальных факторов, которые мы не учли в модели (район, этаж, состояние дома, капризы рынка и т.д.). Природа заложила в неё всё то, что не объясняется одной лишь площадью.

Модель $(1.1)$ утверждает, что реальная цена $y$ складывается из систематической части $w_0 + w_1 x$ (детерминированной и одинаковой для всех объектов с данной площадью) и случайной добавки $\varepsilon$, уникальной для каждой квартиры. Поскольку мы не знаем и никогда не узнаем все неучтённые факторы, мы относим их к случайной ошибке.

**Наша цель** — имея набор наблюдений $(x_1, y_1), (x_2, y_2), \dots, (x_n, y_n)$, найти такие значения параметров $w_0$ и $w_1$, чтобы предсказания $\hat{y}_i = w_0 + w_1 x_i$ были как можно ближе к реальным $y_i$. Прямая $\hat{y} = w_0 + w_1 x$ называется **линией регрессии**.  






## 2. Перебор возможных прямых-кандидатов

Предположим, что нам неизвестны истинные значения параметров $w_0$ и $w_1$. Для их определения мы могли бы рассмотреть **все возможные прямые** вида $\hat{y} = w_0 + w_1 x$ и выбрать среди этого семейства такую, которая лучше всего приближает имеющиеся данные.

Отобразим на графике множество прямых-кандидатов, перебирая различные наклоны $w_1$ и сдвиги $w_0$:


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Генерация данных (квартиры) ---
np.random.seed(42)
n = 40
x = np.random.uniform(30, 150, n)           # площадь, м²
true_w0, true_w1 = 20, 3.5                  # истинные параметры (неизвестны в реальности)
noise = np.random.normal(0, 30, n)          # случайный шум
y = true_w0 + true_w1 * x + noise           # наблюдаемые цены

# --- Рисунок 1: Множество возможных прямых-кандидатов ---
plt.figure(figsize=(10, 6))
plt.scatter(x, y, alpha=0.7, label='Наблюдения (квартиры)')

# Перебор множества прямых-кандидатов
for w1 in np.arange(0, 7, 0.5):            # наклон от 0 до 7
    for w0 in np.arange(-50, 151, 50):      # сдвиг от -50 до 150
        y_predicted = w0 + w1 * x
        plt.plot(x, y_predicted, color='red', alpha=0.1, linewidth=0.8)

plt.xlabel('Площадь, м²')
plt.ylabel('Цена, тыс. долл.')
plt.title('Рисунок 1: Множество возможных прямых-кандидатов')
plt.legend()
plt.grid(alpha=0.3)
plt.xlim(25, 155)
plt.tight_layout()
plt.show()



На рисунке 1 изображено семейство прямых, каждая из которых является одной из гипотез о связи площади и цены. Среди этого множества нам предстоит выбрать ту, которая наилучшим образом соответствует данным.


## 3. Сравнение прямых: какая подходит лучше?

Чтобы понять, какая прямая лучше, выделим несколько характерных кандидатов и сравним их между собой. Одни прямые слишком крутые, другие — слишком пологие, третьи проходят слишком высоко или слишком низко относительно облака точек.



> Код ниже повторяет генерацию данных для самодостаточности фрагмента».
Или вынести генерацию в начало и ссылаться на неё



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import inv

np.random.seed(42)
n = 40
x = np.random.uniform(30, 150, n)
true_w0, true_w1 = 20, 3.5
noise = np.random.normal(0, 30, n)
y = true_w0 + true_w1 * x + noise

# МНК-оценки коэффициентов (оптимальная прямая)
X_design = np.vstack([np.ones_like(x), x]).T
w_hat = inv(X_design.T @ X_design) @ X_design.T @ y
w0_opt, w1_opt = w_hat[0], w_hat[1]

# Три неоптимальные прямые для иллюстрации
w0_bad1, w1_bad1 = 100, 1.5    # слишком пологая
w0_bad2, w1_bad2 = -50, 5.0    # слишком крутая
w0_bad3, w1_bad3 = 150, 2.0    # слишком высокая

x_line = np.linspace(25, 155, 200)

plt.figure(figsize=(10, 6))
plt.scatter(x, y, alpha=0.7, label='Наблюдения (квартиры)')

# Неоптимальные прямые
plt.plot(x_line, w0_bad1 + w1_bad1 * x_line, '--', color='orange', linewidth=2, label='Слишком пологая')
plt.plot(x_line, w0_bad2 + w1_bad2 * x_line, '--', color='green', linewidth=2, label='Слишком крутая')
plt.plot(x_line, w0_bad3 + w1_bad3 * x_line, '--', color='purple', linewidth=2, label='Слишком высокая')

# Оптимальная прямая (МНК)
plt.plot(x_line, w0_opt + w1_opt * x_line, 'r-', linewidth=2.5, label='Наилучшая прямая (МНК)')

# Вертикальные отрезки ошибок для оптимальной прямой
y_opt_line = w0_opt + w1_opt * x
for i in range(n):
    plt.plot([x[i], x[i]], [y[i], y_opt_line[i]], color='gray', alpha=0.5, linewidth=0.8)

plt.xlabel('Площадь, м²')
plt.ylabel('Цена, тыс. долл.')
plt.title('Рисунок 2: Облако точек и несколько прямых-кандидатов')
plt.legend()
plt.grid(alpha=0.3)
plt.xlim(25, 155)
plt.tight_layout()
plt.show()


На рисунке 2 изображено типичное облако точек и несколько вариантов прямой:
- **Слишком пологая** (оранжевая) — систематически занижает цены для больших квартир и завышает для маленьких.
- **Слишком крутая** (зелёная) — переоценивает влияние площади, давая большие ошибки.
- **Слишком высокая** (фиолетовая) — проходит выше большинства точек, завышая предсказания.
- **Наилучшая прямая** (красная) — найдена методом наименьших квадратов, она проходит «в гуще» точек.

Вертикальные серые отрезки показывают ошибки предсказания $|y_i - \hat{y}_i|$ для оптимальной прямой. Интуитивно мы выбираем ту прямую, у которой эти отклонения в среднем минимальны.




## 2. Метод наименьших квадратов (МНК) — первое знакомство

Для каждого наблюдения $i$ определим **остаток** (residual) — разницу между реальным откликом и предсказанием модели:

$$
e_i = y_i - \hat{y}_i = y_i - (w_0 + w_1 x_i). \tag{2.1}
$$

Если остаток положителен, реальная цена оказалась выше предсказанной; если отрицателен — ниже. Идеальная прямая должна давать маленькие остатки.

Первая мысль: сложить все $e_i$ и минимизировать сумму. Но $\sum e_i$ не годится, потому что положительные и отрицательные отклонения компенсируют друг друга. Можно получить сумму, близкую к нулю, даже если линия проходит далеко от многих точек — достаточно, чтобы отклонения вверх и вниз уравновесились. Нужна мера, которая штрафует любое отклонение независимо от знака.

Естественный кандидат — **сумма модулей ошибок** $\sum |e_i|$. Это дало бы метод наименьших модулей (Least Absolute Deviations). Однако у модуля есть недостатки, делающие его неудобным для массового использования:
- Функция $|e|$ имеет излом в точке $e=0$, её производная там не определена. Это усложняет поиск минимума аналитическими и численными методами.
- Модуль одинаково строг к малым и большим ошибкам: увеличение ошибки с 1 до 2 штрафуется так же, как с 99 до 100. Во многих практических задачах мы хотим *сильнее* наказывать грубые промахи, чтобы один «выброс» не уводил прямую слишком далеко от основной массы точек. Квадратичная функция это обеспечивает.

Поэтому классическая линейная регрессия минимизирует **сумму квадратов ошибок** (Sum of Squared Errors, SSE):

$$
SSE(w_0, w_1) = \sum_{i=1}^{n} e_i^2 = \sum_{i=1}^{n} (y_i - w_0 - w_1 x_i)^2. \tag{2.2}
$$

Возведение в квадрат решает обе проблемы: функция гладкая (бесконечно дифференцируемая), выпуклая, и круто возрастает при больших отклонениях, так что единственная большая ошибка вносит в SSE гораздо больший вклад, чем несколько маленьких. Это приводит к тому, что МНК-линия чувствительна к выбросам, но в предположении нормальных ошибок такое поведение статистически оптимально (как мы увидим в разделе 3).



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

e = np.linspace(-2, 2, 200)
abs_e = np.abs(e)
square_e = e**2

plt.figure(figsize=(8, 5))
plt.plot(e, abs_e, label='|e| (модуль)', color='blue', linewidth=2)
plt.plot(e, square_e, label='e² (квадрат)', color='red', linewidth=2)
plt.xlabel('Ошибка e')
plt.ylabel('Значение функции')
plt.title('Рисунок 3: Сравнение функций потерь |e| и e²')
plt.legend()
plt.grid(alpha=0.3)
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.text(0.5, 1.5, 'Гладкая, выпуклая', color='red', fontsize=10)
plt.text(-1.8, 1.2, 'Излом в нуле', color='blue', fontsize=10)
plt.tight_layout()
plt.savefig('figure2_abs_vs_square.png', dpi=150)
plt.show()



Иногда удобнее оперировать **среднеквадратичной ошибкой** (Mean Squared Error, MSE):

$$
MSE(w_0, w_1) = \frac{1}{n} SSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - w_0 - w_1 x_i)^2. \tag{2.3}
$$

MSE имеет понятную размерность (квадрат единицы измерения $y$) и усредняет ошибку по числу наблюдений, что позволяет сравнивать модели на выборках разного размера.

**Ключевой момент:** деление на положительную константу $n$ не сдвигает точку минимума. Если некоторая пара $(w_0, w_1)$ минимизирует SSE, она же минимизирует и MSE, и наоборот. Положение минимума зависит только от формы функции, а умножение на константу лишь меняет масштаб по вертикали. Поэтому при выводе оптимальных коэффициентов мы можем работать с SSE, не беспокоясь о множителе $1/n$. Это также означает, что МНК-оценки одновременно являются и оценками, минимизирующими среднеквадратичную ошибку.

Итак, перед нами оптимизационная задача:

$$
(\hat w_0, \hat w_1) = \underset{w_0, w_1}{\arg\min} \sum_{i=1}^{n} (y_i - w_0 - w_1 x_i)^2. \tag{2.4}
$$

Но почему мы вообще минимизируем именно *сумму квадратов*? Является ли это просто удобной математической уловкой или за этим стоит нечто более фундаментальное? Обратимся к теории вероятностей.

## 3. Вероятностное обоснование: принцип максимального правдоподобия

В модели $y = w_0 + w_1 x + \varepsilon$ мы выделили систематическую часть и случайную добавку. Теперь нужно сделать предположение о том, как распределена эта случайная ошибка $\varepsilon$. Разумный и самый популярный выбор — нормальное (гауссово) распределение. Причин несколько:
- **Центральная предельная теорема (ЦПТ).** Ошибка $\varepsilon$ складывается из множества мелких неучтённых факторов (этаж, шум соседей, качество отделки, переговоры сторон и пр.). Если эти факторы примерно независимы и каждый вносит небольшой вклад, их сумма стремится к нормальному распределению, даже если сами отдельные факторы распределены иначе.
- **Математическое удобство.** Нормальное распределение имеет лёгкие для работы аналитические свойства: произведение плотностей превращается в сумму квадратов, что приводит к МНК.
- **Интерпретируемость.** Параметры нормального распределения (среднее и дисперсия) имеют прямой физический смысл.

Итак, мы принимаем, что ошибки независимы и одинаково распределены (i.i.d.):

$$
\varepsilon_i \sim \mathcal{N}(0, \sigma^2), \quad i = 1, \dots, n,
$$

где $\sigma^2 > 0$ — дисперсия ошибки, характеризующая типичный разброс вокруг линии регрессии.
- $\mathbb{E}[\varepsilon_i] = 0$ означает, что в среднем модель не ошибается: систематическая часть $w_0 + w_1 x$ действительно является *условным математическим ожиданием* $y$ при данном $x$.
- Равенство $\mathrm{Var}(\varepsilon_i) = \sigma^2$ для **всех** наблюдений называют **гомоскедастичностью**: дисперсия ошибки постоянна и не зависит ни от номера наблюдения, ни от величины $x$. Иными словами, разброс цен вокруг линии регрессии одинаков для маленьких, средних и больших квартир. Это — ключевое предположение классической модели, и его проверка составляет важную часть диагностики (см. часть II).

Поскольку $w_0 + w_1 x$ при фиксированном $x$ является константой, из нормальности $\varepsilon$ немедленно следует, что условное распределение отклика тоже нормально:

$$
y \mid x \;\sim\; \mathcal{N}\bigl(w_0 + w_1 x,\; \sigma^2\bigr). \tag{3.1}
$$

Это означает, что для любого заданного $x$ возможные значения $y$ группируются вокруг прямой, а вероятность отклонения быстро убывает с его величиной. Самая вероятная цена — это $\hat{y} = w_0 + w_1 x$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Возьмём ту же МНК-прямую и дисперсию ошибок
sigma = np.std(y - (w0_opt + w1_opt * x), ddof=2)

x_line = np.linspace(25, 155, 200)
y_line = w0_opt + w1_opt * x_line

# Выберем три характерные точки по x
x_points = np.array([50, 90, 130])
y_means = w0_opt + w1_opt * x_points

plt.figure(figsize=(10, 6))
# Фоновая прямая
plt.plot(x_line, y_line, 'k-', linewidth=2, label='Линия регрессии')
# Разброс ±2σ
plt.fill_between(x_line, y_line - 2*sigma, y_line + 2*sigma, color='gray', alpha=0.15, label='±2σ')

# Рисуем колоколы плотности для трёх точек
scale_factor = 15   # масштабный коэффициент для наглядности
for x0, y0 in zip(x_points, y_means):
    y_vals = np.linspace(y0 - 3*sigma, y0 + 3*sigma, 200)
    pdf = norm.pdf(y_vals, loc=y0, scale=sigma)
    # Смещаем колокол по горизонтали: к x0 добавляем pdf * scale_factor
    plt.plot(x0 + pdf * scale_factor, y_vals, color='blue', linewidth=2)
    # Вертикальная линия от x0 до графика
    plt.plot([x0, x0 + pdf.max() * scale_factor], [y0, y0], 'k--', linewidth=1, alpha=0.5)
    # Точка центра
    plt.plot(x0, y0, 'ro')

plt.scatter([], [], color='blue', linewidth=2, label='Условное распределение y|x')
plt.xlabel('Площадь, м²')
plt.ylabel('Цена, тыс. долл.')
plt.title('Рисунок 4: Условные нормальные распределения вокруг линии регрессии')
plt.legend()
plt.grid(alpha=0.3)
plt.xlim(30, 150)
plt.tight_layout()
plt.show()

> **Примечание по поводу `sigma = np.std(y - (w0_opt + w1_opt * x), ddof=2)`.**  
> Здесь мы вычисляем **несмещённую** оценку среднеквадратического отклонения ошибок. Параметр `ddof=2` означает, что из знаменателя вычитается 2 — именно столько степеней свободы теряется при оценке двух параметров модели ($w_0$ и $w_1$). Если бы мы использовали стандартное `ddof=0` (поведение по умолчанию в некоторых функциях), то получили бы **смещённую** оценку $\hat{\sigma}_{\text{MLE}}$ (максимального правдоподобия):  
$ \hat{\sigma}_{\text{MLE}} = \sqrt{\frac{SSE}{n}}. $  
Несмещённая оценка имеет вид:  $ s = \sqrt{\frac{SSE}{n-2}}. $  
> Для наглядной визуализации «колоколов» разница между $s$ и $\hat{\sigma}_{\text{MLE}}$ обычно незначительна, но в статистических выводах (доверительные интервалы, проверка гипотез) важно использовать именно несмещённую оценку $s$. Более подробно вопрос смещённости разбирается в параграфе, посвящённом методу максимального правдоподобия (см. формулу для $\hat{\sigma}^2_{\text{MLE}}$ и последующее обсуждение степеней свободы).




Теперь применим **принцип максимального правдоподобия** (Maximum Likelihood Estimation, MLE). Идея проста: из всех возможных значений параметров $(w_0, w_1, \sigma^2)$ мы выберем те, при которых наблюдать нашу конкретную выборку было бы наиболее вероятно.

Для независимых наблюдений совместная плотность вероятности (функция правдоподобия) равна произведению индивидуальных плотностей:

$$
L(w_0, w_1, \sigma^2) = \prod_{i=1}^{n} \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left( -\frac{(y_i - w_0 - w_1 x_i)^2}{2\sigma^2} \right). \tag{3.2}
$$

Работать напрямую с произведением экспонент неудобно. Возьмём натуральный логарифм — он монотонно возрастает, поэтому точка максимума не изменится. Логарифмическая функция правдоподобия (log-likelihood):

$$
\ell(w_0, w_1, \sigma^2) = \ln L = \sum_{i=1}^{n} \left[ -\frac{1}{2}\ln(2\pi\sigma^2) - \frac{(y_i - w_0 - w_1 x_i)^2}{2\sigma^2} \right].
$$

Вынесем константы за знак суммы:

$$
\ell = -\frac{n}{2}\ln(2\pi) - \frac{n}{2}\ln(\sigma^2) - \frac{1}{2\sigma^2}\underbrace{\sum_{i=1}^{n} (y_i - w_0 - w_1 x_i)^2}_{SSE}. \tag{3.3}
$$

Теперь посмотрим, как $\ell$ зависит от $w_0, w_1$. Первые два слагаемых от них не зависят, а третье — это отрицательная константа $-\frac{1}{2\sigma^2}$, умноженная на SSE. Чтобы $\ell$ была максимальной, вычитаемое должно быть как можно меньше, то есть SSE должна быть минимальной:

$$
\max_{w_0, w_1} \ell \quad \Longleftrightarrow \quad \min_{w_0, w_1} SSE.
$$

**Ключевой вывод:** в предположении нормальности, независимости и гомоскедастичности ошибок, метод наименьших квадратов и метод максимального правдоподобия дают одни и те же оценки для коэффициентов $w_0$ и $w_1$. МНК — не произвольная эвристика, а прямое следствие фундаментального статистического принципа. Это объясняет, почему оценки МНК наследуют желаемые свойства MLE: состоятельность, асимптотическую эффективность и асимптотическую нормальность. Более того, теперь мы можем строить доверительные интервалы и проверять гипотезы, опираясь на развитую теорию максимального правдоподобия.

**Углублённый взгляд. Полный вывод MLE для $\sigma^2$.**  
Можно пойти дальше и оценить дисперсию ошибки $\sigma^2$. Продифференцируем $\ell$ по $\sigma^2$ (рассматривая $\sigma^2$ как переменную, а не $\sigma$):

$$
\frac{\partial \ell}{\partial \sigma^2} = -\frac{n}{2}\frac{1}{\sigma^2} + \frac{1}{2\sigma^4} SSE = 0.
$$

Умножая на $2\sigma^4$, получаем $-n\sigma^2 + SSE = 0$, откуда

$$
\hat{\sigma}^2_{\text{MLE}} = \frac{SSE}{n} = \frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2.
$$

Эта оценка совпадает с MSE (среднеквадратичной ошибкой). Однако она является смещённой: $\mathbb{E}[\hat{\sigma}^2_{\text{MLE}}] = \frac{n-2}{n}\sigma^2 < \sigma^2$. Причина в том, что при вычислении SSE мы уже использовали данные для оценки двух параметров ($w_0, w_1$), и это «съедает» две степени свободы. **Интуитивно:** мы потратили два «кусочка» информации на подгонку прямой, поэтому для оценки разброса остаётся только $n-2$ независимых отклонений. Несмещённая оценка дисперсии ошибки имеет вид

$$
s^2 = \frac{SSE}{n-2}.
$$

Именно её обычно используют при расчёте стандартных ошибок коэффициентов, в t-тестах и при построении доверительных интервалов. При больших $n$ разница между $\hat{\sigma}^2_{\text{MLE}}$ и $s^2$ несущественна, но на малых выборках поправка на число степеней свободы важна.

Теперь, имея прочный статистический фундамент, мы можем смело приступать к отысканию минимума SSE.

## 4. Аналитическое решение для коэффициентов

Обозначим функцию потерь (cost function), которую нужно минимизировать:

$$
J(w_0, w_1) = SSE = \sum_{i=1}^{n} (y_i - w_0 - w_1 x_i)^2. \tag{4.1}
$$

Это квадратичная форма относительно $w_0$ и $w_1$. Она выпукла и имеет единственную точку глобального минимума. Найдём её, приравняв к нулю частные производные.

**Шаг 1: частная производная по $w_0$.**

$$
\frac{\partial J}{\partial w_0} = \sum_{i=1}^{n} 2(y_i - w_0 - w_1 x_i) \cdot (-1) = -2 \sum_{i=1}^{n} (y_i - w_0 - w_1 x_i).
$$

Приравниваем к нулю (множитель $-2$ можно опустить):

$$
\sum_{i=1}^{n} (y_i - w_0 - w_1 x_i) = 0.
$$

Раскроем сумму:

$$
\sum y_i - n w_0 - w_1 \sum x_i = 0.
$$

Выразим $n w_0$:

$$
n w_0 = \sum y_i - w_1 \sum x_i.
$$

Поделим на $n$ и введём выборочные средние $\bar{x} = \frac{1}{n}\sum x_i$, $\bar{y} = \frac{1}{n}\sum y_i$:

$$
w_0 = \bar{y} - w_1 \bar{x}. \tag{4.2}
$$

Это уравнение имеет простой геометрический смысл: наилучшая прямая всегда проходит через «центр тяжести» данных — точку $(\bar{x}, \bar{y})$.

**Шаг 2: частная производная по $w_1$.**

$$
\frac{\partial J}{\partial w_1} = \sum_{i=1}^{n} 2(y_i - w_0 - w_1 x_i) \cdot (-x_i) = -2 \sum_{i=1}^{n} x_i (y_i - w_0 - w_1 x_i).
$$

Приравниваем к нулю (снова опускаем $-2$):

$$
\sum_{i=1}^{n} x_i (y_i - w_0 - w_1 x_i) = 0. \tag{4.3}
$$

Подставим $w_0$ из $(4.2)$ в $(4.3)$:

$$
\sum_{i=1}^{n} x_i \bigl(y_i - (\bar{y} - w_1 \bar{x}) - w_1 x_i\bigr) = \sum_{i=1}^{n} x_i \bigl((y_i - \bar{y}) - w_1 (x_i - \bar{x})\bigr) = 0.
$$

Раскроем скобки:

$$
\sum x_i (y_i - \bar{y}) - w_1 \sum x_i (x_i - \bar{x}) = 0.
$$

Полезный приём: добавим и вычтем $\bar{x}$ в первых сомножителях. Заметим, что

$$
\sum x_i (y_i - \bar{y}) = \sum (x_i - \bar{x})(y_i - \bar{y}) + \bar{x} \sum (y_i - \bar{y}) = \sum (x_i - \bar{x})(y_i - \bar{y}) + 0,
$$
$$
\sum x_i (x_i - \bar{x}) = \sum (x_i - \bar{x})^2 + \bar{x} \sum (x_i - \bar{x}) = \sum (x_i - \bar{x})^2.
$$

Таким образом, уравнение сводится к

$$
\sum (x_i - \bar{x})(y_i - \bar{y}) - w_1 \sum (x_i - \bar{x})^2 = 0.
$$

Отсюда, если $\sum (x_i - \bar{x})^2 \neq 0$ (то есть не все $x_i$ одинаковы — иначе задача вырождена), получаем окончательную формулу для наклона:

$$
w_1 = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n} (x_i - \bar{x})^2}. \tag{4.4}
$$

Числитель — это выборочная ковариация между $x$ и $y$ (умноженная на $n$), знаменатель — выборочная дисперсия $x$ (тоже умноженная на $n$). Таким образом, наклон — это просто отношение «совместной изменчивости» $x$ и $y$ к изменчивости $x$. Если $x$ и $y$ имеют тенденцию вместе отклоняться от своих средних в одну сторону, наклон положителен; если в разные — отрицателен. Если связи нет, числитель близок к нулю, и наклон близок к нулю.

Свободный член $w_0$ затем находится по формуле $(4.2)$.

### Простой числовой пример «вручную»

Рассмотрим три наблюдения: $(x, y) = (1, 2), (2, 3), (3, 5)$. Найдём наилучшую прямую без калькулятора, следуя формулам.

1. Средние: $\bar{x} = \frac{1+2+3}{3} = 2$; $\bar{y} = \frac{2+3+5}{3} = 10/3 \approx 3.333$.
2. Вычислим отклонения:
   - $(x_i-\bar{x})$: $(1-2)=-1,\ (2-2)=0,\ (3-2)=1$.
   - $(y_i-\bar{y})$: $(2-3.333)=-1.333,\ (3-3.333)=-0.333,\ (5-3.333)=1.667$.
3. Числитель $\sum (x_i-\bar{x})(y_i-\bar{y}) = (-1)\cdot(-1.333) + 0\cdot(-0.333) + 1\cdot 1.667 = 1.333 + 1.667 = 3$.
4. Знаменатель $\sum (x_i-\bar{x})^2 = (-1)^2 + 0^2 + 1^2 = 2$.
5. Наклон $w_1 = 3/2 = 1.5$.
6. Свободный член $w_0 = \bar{y} - w_1\bar{x} = 3.333 - 1.5 \cdot 2 = 3.333 - 3 = 0.333$.

Таким образом, уравнение регрессии: $\hat{y} = 0.333 + 1.5 x$.

Вычислим предсказанные значения и остатки для проверки:
- $x=1$: $\hat{y}=0.333+1.5=1.833$, остаток $2-1.833=0.167$.
- $x=2$: $\hat{y}=0.333+3.0=3.333$, остаток $3-3.333=-0.333$.
- $x=3$: $\hat{y}=0.333+4.5=4.833$, остаток $5-4.833=0.167$.

SSE = $0.167^2 + (-0.333)^2 + 0.167^2 \approx 0.0279 + 0.1111 + 0.0279 = 0.1669$. (Небольшие расхождения с первоначальным округлением некритичны.)

Прямая не проходит идеально через точки, но является наилучшей в смысле минимума SSE.

### Интерпретация коэффициентов

- **Наклон $w_1 = 1.5$.** При увеличении $x$ на одну единицу отклик $y$ в среднем возрастает на $1.5$ единицы. В примере с квартирами это означает, что каждый дополнительный квадратный метр площади повышает цену в среднем на 1.5 условных единиц (например, тысяч долларов).
- **Свободный член $w_0 = 0.333$.** Формально это предсказанное значение $y$ при $x = 0$. Однако нулевая площадь лишена физического смысла — квартир с площадью ноль не бывает. Поэтому $w_0$ часто интерпретируют просто как «подпорку», сдвигающую прямую по вертикали так, чтобы она наилучшим образом легла в диапазон наблюдаемых данных. Предметную интерпретацию свободный член имеет только тогда, когда нулевое значение признака достижимо и осмысленно.
- **Осторожность при экстраполяции.** Наша модель обучалась на данных с $x$ от 1 до 3. Она адекватно описывает связь *только* в этом диапазоне. Если мы попытаемся предсказать цену для квартиры площадью 100 кв. м по формуле $\hat{y} = 0.333 + 1.5 \cdot 100 = 150.333$, то получим, что такая квартира стоит как небольшой особняк. Это, скорее всего, бессмысленно, потому что в реальности зависимость между площадью и ценой часто становится нелинейной (насыщение спроса, ограниченный бюджет) или даже меняет знак при очень больших значениях. Экстраполяция за пределы обучающих данных всегда требует осторожности и дополнительного обоснования предметной областью.

## 5. Матричная запись для простой регрессии (один признак)

Когда мы имеем дело с одним признаком, формулы $(4.2)$ и $(4.4)$ прекрасно работают. Но в машинном обучении и статистике принято использовать матричную запись — она компактна, легко обобщается на множество признаков и удобна для вычислений. Покажем, как переписать простую линейную регрессию в матричном виде, и убедимся, что это приводит к тем же самым результатам.

Введём следующие матрицы и векторы:

$$
Y = \begin{pmatrix} y_1 \\ y_2 \\ \vdots \\ y_n \end{pmatrix}_{n\times 1}, \quad
X = \begin{pmatrix} 1 & x_1 \\ 1 & x_2 \\ \vdots & \vdots \\ 1 & x_n \end{pmatrix}_{n\times 2}, \quad
w = \begin{pmatrix} w_0 \\ w_1 \end{pmatrix}_{2\times 1}, \quad
\varepsilon = \begin{pmatrix} \varepsilon_1 \\ \varepsilon_2 \\ \vdots \\ \varepsilon_n \end{pmatrix}_{n\times 1}.
$$

Вектор $Y$ содержит все наблюдения отклика. Матрица $X$ называется **матрицей плана** (design matrix). Она состоит из двух столбцов: первый — единичный столбец, соответствующий свободному члену $w_0$, второй — значения признака $x$. Вектор $w$ объединяет искомые параметры.

Тогда модель для всех наблюдений сразу записывается как

$$
Y = X w + \varepsilon. \tag{5.1}
$$

Сумма квадратов ошибок в матричном виде — это квадрат евклидовой нормы вектора остатков:

$$
SSE(w) = \| Y - X w \|^2 = (Y - X w)^T (Y - X w). \tag{5.2}
$$

**Вывод нормальных уравнений.** Раскроем квадрат нормы:

$$
SSE(w) = Y^T Y - 2 Y^T X w + w^T X^T X w.
$$

Продифференцируем по вектору $w$ (пользуясь правилами матричного дифференцирования) и приравняем к нулю:

$$
\frac{\partial SSE}{\partial w} = -2 X^T Y + 2 X^T X w = 0 \quad \Longrightarrow \quad X^T X w = X^T Y. \tag{5.3}
$$

Это и есть **матричная форма нормальных уравнений**. Размерность: $X^T X$ — матрица $2 \times 2$, $X^T Y$ — вектор $2 \times 1$. Покажем, что $(5.3)$ в точности совпадает с системой, полученной нами из частных производных.

Вычислим матричные произведения явно:

$$
X^T X = \begin{pmatrix} 1 & 1 & \cdots & 1 \\ x_1 & x_2 & \cdots & x_n \end{pmatrix}
\begin{pmatrix} 1 & x_1 \\ 1 & x_2 \\ \vdots & \vdots \\ 1 & x_n \end{pmatrix}
= \begin{pmatrix} n & \sum x_i \\ \sum x_i & \sum x_i^2 \end{pmatrix}.
$$

$$
X^T Y = \begin{pmatrix} 1 & 1 & \cdots & 1 \\ x_1 & x_2 & \cdots & x_n \end{pmatrix}
\begin{pmatrix} y_1 \\ y_2 \\ \vdots \\ y_n \end{pmatrix}
= \begin{pmatrix} \sum y_i \\ \sum x_i y_i \end{pmatrix}.
$$

Таким образом, уравнение $X^T X w = X^T Y$ разворачивается в систему

$$
\begin{cases}
n w_0 + (\sum x_i) w_1 = \sum y_i, \\
(\sum x_i) w_0 + (\sum x_i^2) w_1 = \sum x_i y_i,
\end{cases}
$$

которая с точностью до множителей совпадает с системой нормальных уравнений из раздела 4 (полученной приравниванием частных производных к нулю). Решение существует и единственно, если матрица $X^T X$ обратима. Для простой регрессии это условие сводится к $\det(X^T X) = n \sum x_i^2 - (\sum x_i)^2 \neq 0$, что эквивалентно $\sum (x_i - \bar{x})^2 > 0$, то есть наличию вариации признака $x$. Если все $x_i$ одинаковы, наклон определить невозможно — задача вырождается.

**Явная формула решения.** Если $X^T X$ обратима, решение даётся выражением

$$
\hat{w} = (X^T X)^{-1} X^T Y. \tag{5.4}
$$

Подставим наш числовой пример $(1,2), (2,3), (3,5)$ в матричную форму.

$$
Y = \begin{pmatrix} 2 \\ 3 \\ 5 \end{pmatrix},\quad
X = \begin{pmatrix} 1 & 1 \\ 1 & 2 \\ 1 & 3 \end{pmatrix}.
$$

$$
X^T X = \begin{pmatrix} 3 & 6 \\ 6 & 14 \end{pmatrix},\quad
X^T Y = \begin{pmatrix} 10 \\ 23 \end{pmatrix}.
$$

Определитель $\Delta = 3\cdot 14 - 6\cdot 6 = 42 - 36 = 6$. Обратная матрица:

$$
(X^T X)^{-1} = \frac{1}{6} \begin{pmatrix} 14 & -6 \\ -6 & 3 \end{pmatrix}.
$$

Тогда

$$
\hat{w} = \frac{1}{6} \begin{pmatrix} 14 & -6 \\ -6 & 3 \end{pmatrix}
\begin{pmatrix} 10 \\ 23 \end{pmatrix}
= \frac{1}{6} \begin{pmatrix} 140 - 138 \\ -60 + 69 \end{pmatrix}
= \frac{1}{6} \begin{pmatrix} 2 \\ 9 \end{pmatrix}
= \begin{pmatrix} 1/3 \\ 3/2 \end{pmatrix}.
$$

Мы снова получили $w_0 = 1/3 \approx 0.333$, $w_1 = 1.5$, что совпадает с предыдущими вычислениями. Матричный путь не вводит новых чисел — он лишь переупаковывает вычисления в удобную для компьютера форму.



**Углублённый взгляд. Геометрический смысл.**  

Уравнение $X^T X w = X^T Y$ можно переписать как $X^T (Y - X w) = 0$. Это означает, что вектор остатков $e = Y - X\hat{w}$ ортогонален (перпендикулярен) каждому столбцу матрицы $X$. В нашем случае столбцы — это вектор $\mathbf{1} = (1,1,\dots,1)^T$ и вектор значений признака $x$. Ортогональность этим столбцам даёт в точности нормальные уравнения:  
$$
\sum_{i=1}^n e_i = 0 \quad \text{и} \quad \sum_{i=1}^n x_i e_i = 0.
$$

**Пример для трёх точек.**  
Возьмём данные из числового примера: $(1,2), (2,3), (3,5)$. Матрица $X$ и вектор $Y$:
$$
X = \begin{pmatrix}1 & 1\\1 & 2\\1 & 3\end{pmatrix},\quad Y = \begin{pmatrix}2\\3\\5\end{pmatrix}.
$$
Мы нашли $\hat{w}_0 = 1/3 \approx 0.333$, $\hat{w}_1 = 1.5$. Тогда
$$
\hat{Y} = X\hat{w} = \begin{pmatrix}0.333+1.5\cdot1\\0.333+1.5\cdot2\\0.333+1.5\cdot3\end{pmatrix} = \begin{pmatrix}1.833\\3.333\\4.833\end{pmatrix},
$$
остатки $e = Y - \hat{Y} = (0.167,\;-0.333,\;0.167)$.  
Проверим ортогональность:
$$
e^T \mathbf{1} = 0.167 - 0.333 + 0.167 = 0,\quad
e^T x = 0.167\cdot1 + (-0.333)\cdot2 + 0.167\cdot3 = 0.167 - 0.666 + 0.501 = 0.
$$
Оба скалярных произведения равны нулю. Значит, вектор остатков действительно перпендикулярен плоскости, натянутой на $\mathbf{1}$ и $x$.

**Геометрическая картина в пространстве наблюдений.**  
Каждое наблюдение — это координата в $n$-мерном пространстве (у нас $n=3$). Вектор $Y$ задаёт точку в этом пространстве. Все возможные линейные комбинации столбцов $X$ (т.е. векторы вида $Xw$) образуют двумерную плоскость, проходящую через начало координат. МНК-решение $\hat{Y} = X\hat{w}$ — это **ортогональная проекция** вектора $Y$ на эту плоскость. Вектор остатков $e = Y - \hat{Y}$ перпендикулярен плоскости — это и есть кратчайшее расстояние от точки $Y$ до плоскости.

Такая интерпретация остаётся верной и для любого числа признаков: $\hat{Y}$ всегда является ортогональной проекцией $Y$ на линейное подпространство, порождённое столбцами $X$. Это делает идею МНК не только алгебраической, но и интуитивно понятной геометрически.


In [ ]:
import numpy as np
import plotly.graph_objects as go

# Новые данные: (x, y) = (1,2), (3,6), (4,1)
# Точка (4,1) сильно отклоняется от линейного тренда — остаток будет большим
x_vals = np.array([1, 3, 4])
y_vals = np.array([2, 6, 1])
n = 3

# Матрица X и МНК
ones = np.ones(n)
X = np.column_stack((ones, x_vals))
w = np.linalg.lstsq(X, y_vals, rcond=None)[0]
y_hat = X @ w
residuals = y_vals - y_hat

# Плоскость span(1, x) (расширена, чтобы проходила через начало координат)
u = np.linspace(-0.5, 1.5, 10)
v = np.linspace(-0.5, 4.5, 10)
U, V = np.meshgrid(u, v)
plane_x = U * ones[0] + V * x_vals[0]
plane_y = U * ones[1] + V * x_vals[1]
plane_z = U * ones[2] + V * x_vals[2]

fig = go.Figure()

# Плоскость (полупрозрачная)
fig.add_trace(go.Surface(
    x=plane_x, y=plane_y, z=plane_z,
    opacity=0.3, colorscale='Greys', showscale=False,
    name='Плоскость span(1, x)'
))

# Вектор 1 (синий)
fig.add_trace(go.Scatter3d(
    x=[0, ones[0]], y=[0, ones[1]], z=[0, ones[2]],
    mode='lines+text',
    line=dict(color='blue', width=5),
    text=['', '1'], textposition='top center',
    name='1 (единицы)'
))

# Вектор x (зелёный)
fig.add_trace(go.Scatter3d(
    x=[0, x_vals[0]], y=[0, x_vals[1]], z=[0, x_vals[2]],
    mode='lines+text',
    line=dict(color='green', width=5),
    text=['', 'x'], textposition='top center',
    name='x (признак)'
))

# Вектор Y (красный)
fig.add_trace(go.Scatter3d(
    x=[0, y_vals[0]], y=[0, y_vals[1]], z=[0, y_vals[2]],
    mode='lines+text',
    line=dict(color='red', width=5),
    text=['', 'Y'], textposition='top center',
    name='Y (наблюдения)'
))

# Проекция Ŷ (фиолетовый)
fig.add_trace(go.Scatter3d(
    x=[0, y_hat[0]], y=[0, y_hat[1]], z=[0, y_hat[2]],
    mode='lines+text',
    line=dict(color='purple', width=5),
    text=['', 'Ŷ'], textposition='top center',
    name='Ŷ (проекция)'
))

# Остаток e (ярко-жёлтый, толще, заметнее)
fig.add_trace(go.Scatter3d(
    x=[y_hat[0], y_vals[0]], y=[y_hat[1], y_vals[1]], z=[y_hat[2], y_vals[2]],
    mode='lines+markers+text',
    line=dict(color='gold', width=8),
    marker=dict(size=6, color='gold'),
    text=['', 'e'], textposition='top center',
    name='e = Y - Ŷ (остаток)'
))

# Подсветим точку проекции
fig.add_trace(go.Scatter3d(
    x=[y_hat[0]], y=[y_hat[1]], z=[y_hat[2]],
    mode='markers', marker=dict(size=6, color='purple'),
    showlegend=False
))

# Аннотация с проверкой ортогональности
dot1 = np.dot(residuals, ones)
dot2 = np.dot(residuals, x_vals)
annotation = f'Проверка ортогональности:<br>e·1 = {dot1:.6f}<br>e·x = {dot2:.6f}'
fig.add_annotation(
    x=0.02, y=0.12, xref='paper', yref='paper',
    text=annotation, showarrow=False,
    bgcolor='white', bordercolor='black', borderwidth=1,
    font=dict(size=12)
)

# Настройки сцены
fig.update_layout(
    title='Рисунок 5: Геометрический смысл МНК — проекция Y на плоскость столбцов X<br>'
          'Остаток e (жёлтый) перпендикулярен плоскости (проверьте вращением)',
    scene=dict(
        xaxis_title='координата 1-го наблюдения',
        yaxis_title='координата 2-го наблюдения',
        zaxis_title='координата 3-го наблюдения',
        aspectmode='manual',
        aspectratio=dict(x=1.2, y=1.2, z=1.5),
        camera=dict(eye=dict(x=1.2, y=1.2, z=1.5))
    ),
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.7)'),
    width=900, height=700
)

fig.show()
# Сохранить как интерактивный HTML
fig.write_html('mnk_geometry_clear.html')
print("Интерактивный график сохранён в 'mnk_geometry_clear.html'")


Теперь, когда коэффициенты найдены, встаёт вопрос: насколько полученная модель хороша? Для ответа служит коэффициент детерминации.

## 6. Оценка качества модели: коэффициент детерминации $R^2$

После подгонки прямой мы хотим измерить, какую долю изменчивости (вариации) зависимой переменной объяснила модель. Естественная мера — коэффициент детерминации $R^2$.

Введём три суммы квадратов, характеризующие разброс данных:

- **SST (Total Sum of Squares)** — полная вариация $y$ относительно своего среднего:

$$
SST = \sum_{i=1}^{n} (y_i - \bar{y})^2.
$$

Эта величина отражает общий разброс цен, независимо от модели.

- **SSE (Error Sum of Squares)** — сумма квадратов остатков, то есть вариация, которую модель *не смогла* объяснить:

$$
SSE = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2.
$$

- **SSR (Regression Sum of Squares)** — вариация, объяснённая моделью. Её можно вычислить как $SSR = SST - SSE$. SSR показывает, насколько предсказания $\hat{y}_i$ отклоняются от среднего $\bar{y}$; можно также записать $SSR = \sum (\hat{y}_i - \bar{y})^2$.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Данные из числового примера лекции (1,2), (2,3), (3,5)
x_points = np.array([1, 2, 3])
y_points = np.array([2, 3, 5])

# Расчёт линии регрессии
x_mean = np.mean(x_points)
y_mean = np.mean(y_points)
w1 = np.sum((x_points - x_mean) * (y_points - y_mean)) / np.sum((x_points - x_mean)**2)
w0 = y_mean - w1 * x_mean

# Для точки x0 = 3, y0 = 5
x0, y0 = 3, 5
y_pred = w0 + w1 * x0

plt.figure(figsize=(8, 5))
# Линия регрессии
x_line = np.linspace(0.5, 3.5, 100)
plt.plot(x_line, w0 + w1 * x_line, 'r-', label='Линия регрессии')
# Точки данных
plt.scatter(x_points, y_points, color='black', zorder=5, label='Данные')
# Выделенная точка
plt.scatter(x0, y0, color='blue', s=80, zorder=6, edgecolors='black', label='Рассматриваемая точка')
# Горизонтальная линия среднего
plt.axhline(y=y_mean, color='green', linestyle='--', label=f'Среднее ȳ = {y_mean:.2f}')

# Отрезки отклонений
plt.plot([x0, x0], [y_mean, y0], color='purple', linestyle='-', linewidth=2,
         label='Общее отклонение (yᵢ - ȳ)')
plt.plot([x0, x0], [y_mean, y_pred], color='orange', linestyle='-', linewidth=2,
         label='Объяснённое (ŷᵢ - ȳ)')
plt.plot([x0, x0], [y_pred, y0], color='brown', linestyle='-', linewidth=2,
         label='Остаток (yᵢ - ŷᵢ)')

plt.xlabel('x')
plt.ylabel('y')
plt.title('Рисунок 6: Декомпозиция отклонения для одной точки\n(общее = объяснённое + остаток)')
plt.legend()
plt.grid(alpha=0.3)
plt.xlim(0.5, 3.5)
plt.ylim(0, 6)
plt.tight_layout()
plt.savefig('figure4_decomposition.png', dpi=150)
plt.show()



**Пояснение к рисунку 6.** Благодаря тому, что остатки ортогональны предсказанным значениям (и, в частности, их среднему), общая сумма квадратов распадается на сумму объяснённой и необъяснённой частей: $SST = SSR + SSE$. Это фундаментальное тождество лежит в основе определения $R^2$.

Коэффициент детерминации определяется как доля объяснённой вариации в полной вариации:

$$
R^2 = \frac{SSR}{SST} = 1 - \frac{SSE}{SST} = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}. \tag{6.1}
$$

**Свойства $R^2$ для модели со свободным членом** (наш случай):

- Если модель не даёт ничего сверх простого среднего, то $\hat{y}_i \approx \bar{y}$, тогда $SSE \approx SST$ и $R^2 \approx 0$.
- Если модель идеально предсказывает все точки ($SSE = 0$), то $R^2 = 1$.
- В рамках обучающей выборки и при наличии константы всегда выполнено $0 \le R^2 \le 1$. Это следует из того, что $SSE \le SST$ (доказывается через ортогональность остатков и вектора константы).
- Если модель не содержит свободного члена (принудительно $w_0 = 0$), то $R^2$ может стать отрицательным. То же самое может произойти при вычислении $R^2$ на новых, тестовых данных: если модель на тесте работает хуже, чем простое предсказание средним по обучающей выборке, $SSE > SST$ и $R^2 < 0$.

**Полезное соотношение.** В простой линейной регрессии с константой $R^2$ равен квадрату выборочного коэффициента корреляции Пирсона $r_{xy}$:

$$
R^2 = r_{xy}^2 = \left( \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}} \right)^2.
$$

Это даёт ещё одну интерпретацию: $R^2$ показывает, насколько сильна *линейная* связь между $x$ и $y$. Если точки ложатся точно на прямую, $|r_{xy}| = 1$ и $R^2 = 1$; если разброс хаотичен, $r_{xy} \approx 0$ и $R^2$ близок к нулю.

**Учебный пример с низким $R^2$.**  
Рассмотрим три точки: $(1, 2), (2, 5), (3, 1)$. Здесь средние $\bar{x} = 2$, $\bar{y} = (2+5+1)/3 = 8/3 \approx 2.667$. Вычислим необходимые суммы:

- $(x_i-\bar{x})$: $(1-2)=-1,\ (2-2)=0,\ (3-2)=1$.
- $(y_i-\bar{y})$: $(2-2.667)=-0.667,\ (5-2.667)=2.333,\ (1-2.667)=-1.667$.

Числитель для наклона: $\sum (x_i-\bar{x})(y_i-\bar{y}) = (-1)\cdot(-0.667) + 0\cdot2.333 + 1\cdot(-1.667) = 0.667 - 1.667 = -1.0$.
Знаменатель: $\sum (x_i-\bar{x})^2 = 1+0+1=2$.
Наклон $w_1 = -1.0 / 2 = -0.5$.
Свободный член $w_0 = \bar{y} - w_1\bar{x} = 2.667 - (-0.5)\cdot 2 = 2.667 + 1 = 3.667$.

Предсказания: $\hat{y}_1 = 3.667 -0.5\cdot1 = 3.167$, $\hat{y}_2 = 3.667 -1 = 2.667$, $\hat{y}_3 = 3.667 -1.5 = 2.167$.
Остатки: $2-3.167 = -1.167$, $5-2.667 = 2.333$, $1-2.167 = -1.167$.
SSE = $1.167^2 + 2.333^2 + 1.167^2 \approx 1.361 + 5.444 + 1.361 = 8.166$.
SST = $(2-2.667)^2 + (5-2.667)^2 + (1-2.667)^2 = 0.444 + 5.444 + 2.778 = 8.666$.
Тогда $R^2 = 1 - 8.166/8.666 \approx 1 - 0.942 = 0.058$.

Таким образом, модель объясняет лишь около 6% вариации. Это отрезвляющий пример: иногда прямая — не лучший способ описать данные.

**Важное предостережение:** высокий $R^2$ не является сертификатом адекватности модели. Можно получить $R^2 = 0.9$ для явно нелинейной зависимости, если облако точек случайно напоминает прямую. И наоборот, низкий $R^2$ не всегда означает, что модель бесполезна — например, в социальных науках $R^2 = 0.3$ может считаться отличным результатом, если влияние признака статистически значимо и интерпретируемо. Поэтому $R^2$ всегда следует дополнять диагностикой остатков (см. часть II) и, при необходимости, проверкой гипотез о коэффициентах.

**Углублённый взгляд. Скорректированный $R^2$.**  
В многофакторных моделях (которых мы пока не касаемся) добавление любого нового признака, даже шумового, не уменьшает, а чаще всего увеличивает обычный $R^2$ — просто потому, что модель получает больше степеней свободы для подгонки. Чтобы штрафовать за излишнюю сложность, используют скорректированный $R^2$:

$$
R^2_{\text{adj}} = 1 - \frac{SSE/(n-p-1)}{SST/(n-1)},
$$

где $p$ — число признаков (для простой регрессии $p=1$). При добавлении бесполезных признаков $R^2_{\text{adj}}$ может даже уменьшаться. В простой регрессии эта проблема стоит не остро, но знать о таком показателе полезно.

Теперь, пройдя путь от интуиции до вычисления коэффициентов и оценки качества, важно сформулировать те предположения, на которых держится весь построенный аппарат.

## 7. Сводка предположений классической линейной регрессии

Классическая линейная регрессия опирается на ряд предположений о данных. Их выполнение гарантирует корректность МНК-оценок, их статистические свойства и достоверность выводов (тестов, доверительных интервалов). Различают предположения, необходимые для собственно оценивания, и более сильные предположения, нужные для статистических выводов.

### Пять ключевых предположений

1. **Линейность по параметрам (правильная спецификация модели).**  
   Математическое ожидание отклика является линейной функцией параметров:  
   $$
   \mathbb{E}[y_i] = w_0 + w_1 x_i.
   $$  
   Важно: линейность требуется именно по параметрам $w_0, w_1$, а не по самим признакам. Например, модель $y = w_0 + w_1 x^2$ тоже линейна по параметрам. Нарушение этого предположения (пропуск существенных нелинейных членов) ведёт к смещённым оценкам и неверным предсказаниям. Диагностируется графиком остатков против $x$ или $\hat{y}$.

2. **Независимость ошибок.**  
   Ошибки разных наблюдений некоррелированы: $\mathrm{Cov}(\varepsilon_i, \varepsilon_j) = 0$ при $i \neq j$. Это предположение нарушается, например, во временных рядах (автокорреляция), при кластеризованных данных (цены квартир в одном доме). Следствия: МНК-оценки остаются несмещёнными, но стандартные ошибки смещаются (обычно занижаются), что делает тесты некорректными. Проверяется тестом Дарбина–Уотсона или анализом автокорреляционной функции остатков.

3. **Нулевое математическое ожидание ошибки.**  
   $\mathbb{E}[\varepsilon_i] = 0$ для всех $i$. Это условие по сути не ограничительно, если в модель включён свободный член: любое систематическое смещение будет поглощено $w_0$. Если константы в модели нет, нарушение ведёт к смещению оценок всех коэффициентов.

4. **Гомоскедастичность (постоянство дисперсии ошибок).**  
   $\mathrm{Var}(\varepsilon_i) = \sigma^2$ для всех $i$. Дисперсия ошибки не зависит от $x$ и от номера наблюдения. При нарушении (гетероскедастичности) МНК-оценки остаются несмещёнными, но теряют в эффективности, и, что важнее, стандартные ошибки коэффициентов становятся смещёнными, разрушая t-тесты и доверительные интервалы. Диагностируется графиком остатков против предсказанных значений — веерообразная форма указывает на непостоянство дисперсии. Лечится взвешенным МНК или робастными стандартными ошибками (Уайта, Хьюбера–Уайта).

5. **Нормальность ошибок** (необходима для точных статистических выводов).  
   $\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$. В сочетании с независимостью это даёт точное распределение оценок, на основе которого строятся t- и F-тесты и доверительные интервалы в выборках любого размера. Без нормальности, но при наличии большой выборки, центральная предельная теорема обеспечивает приближённую нормальность оценок, и тесты асимптотически верны. На малых выборках нарушение нормальности может исказить p-значения. Проверяется Q-Q-plot остатков или формальными тестами (Шапиро–Уилка, Колмогорова–Смирнова). В случае проблем можно использовать бутстрап или непараметрические методы.

### Теорема Гаусса–Маркова (BLUE)

При выполнении предположений 1–4 (линейность, независимость, нулевое среднее, гомоскедастичность) МНК-оценки $\hat{w}_0, \hat{w}_1$ являются **наилучшими линейными несмещёнными оценками** (Best Linear Unbiased Estimators, BLUE). Это означает:
- **Линейность:** оценки суть линейные комбинации наблюдений $y_i$.
- **Несмещённость:** $\mathbb{E}[\hat{w}_j] = w_j$ ($j=0,1$).
- **Наилучшие:** среди всех линейных несмещённых оценок они обладают наименьшей дисперсией.

Нормальность ошибок для BLUE не требуется. Если же нормальность выполняется, МНК-оценки дополнительно совпадают с MLE и обладают всеми оптимальными свойствами последних (в том числе асимптотической эффективностью).

### Кратко: что бывает при нарушениях каждого предположения

| Нарушение | Влияние на оценки коэффициентов | Влияние на стандартные ошибки и тесты |
|-----------|--------------------------------|---------------------------------------|
| Нелинейность | Систематическое смещение, неадекватные предсказания | — |
| Зависимость ошибок | Оценки несмещены, но могут быть неэффективны | Стандартные ошибки смещены (часто занижены), тесты неверны |
| Ненулевое среднее ошибок | Смещается $w_0$; если свободный член отсутствует, смещаются и остальные коэффициенты | — |
| Гетероскедастичность | Оценки несмещены, но неэффективны | Стандартные ошибки смещены, t-тесты ошибочны |
| Ненормальность (малая выборка) | Оценки несмещены (при выполнении 1–4) | t- и F-тесты могут давать неверные p-значения; на больших выборках асимптотически допустимы |

### Анонс второй части главы

В следующей части мы детально разберём инструменты диагностики модели: как строить и интерпретировать графики остатков (остатки против предсказанных значений, Q-Q plot), как проверять гомоскедастичность тестами Бройша–Пагана и Уайта, как выявлять влиятельные наблюдения с помощью расстояния Кука. Также обсудим, что делать при обнаружении нарушений — преобразования переменных, взвешенный МНК, робастные стандартные ошибки. Всё это превратит формально построенную прямую в надёжный инструмент анализа данных.

---

## Вопросы для самопроверки

### Для начинающих

1. Запишите уравнение простой линейной регрессии и поясните, зачем в нём слагаемое $\varepsilon$. Почему нельзя просто сказать, что $y$ точно равен $w_0 + w_1 x$?
2. Почему в качестве меры ошибки мы используем квадраты, а не модули? Какие два главных преимущества даёт возведение в квадрат?
3. Дайте определения SSE и MSE. Если объём выборки $n$ увеличивается, а коэффициенты $w_0, w_1$ фиксированы, как изменятся SSE и MSE? Верно ли, что параметры, минимизирующие MSE, те же, что минимизируют SSE?
4. Какое вероятностное распределение мы предположили для ошибок и на основании какой математической теоремы? Что такое гомоскедастичность и как бы она выглядела на графике остатков?
5. Опишите идею метода максимального правдоподобия своими словами. Почему максимизация логарифма правдоподобия приводит к тому же результату, что и минимизация SSE?
6. Вычислите коэффициенты регрессии для точек $(2,4), (3,6), (5,9)$ вручную. Какой получился наклон и свободный член? Проверьте, что прямая проходит через точку средних.
7. Объясните «на пальцах» геометрический смысл матричного решения: что значит, что $\hat{Y}$ — это ортогональная проекция $Y$?
8. Что такое $R^2$ и как он связан с коэффициентом корреляции Пирсона? Может ли $R^2$ равняться нулю или единице? В каком случае он мог бы стать отрицательным на тестовой выборке?

### Для профессионалов

1. Выведите формулы для дисперсий оценок $\hat{w}_0$ и $\hat{w}_1$ в простой линейной регрессии. *(Подсказка: используйте выражение $\hat{w}_1 = \frac{\sum (x_i-\bar{x})y_i}{\sum (x_i-\bar{x})^2}$ и свойство $\text{Var}(\sum c_i y_i) = \sum c_i^2 \text{Var}(y_i)$ при независимых $y_i$.)* Как эти дисперсии зависят от дисперсии ошибок $\sigma^2$, разброса $x$ и размера выборки?
2. Покажите, что оценка $\hat{\sigma}^2_{\text{MLE}} = SSE/n$ является смещённой, вычислив $\mathbb{E}[SSE]$ и указав, сколько степеней свободы теряется. Почему несмещённая оценка имеет вид $s^2 = SSE/(n-2)$?
3. Выведите матричные нормальные уравнения $X^T X w = X^T Y$ непосредственно из минимизации квадратичной формы $(Y - Xw)^T (Y - Xw)$, пользуясь правилами матричного дифференцирования. При каком условии матрица $X^T X$ обратима в случае простой регрессии?
4. Докажите, что при предположениях 1–4 теоремы Гаусса–Маркова МНК-оценка наклона $w_1$ является несмещённой: $\mathbb{E}[\hat{w}_1] = w_1$. (Подсказка: подставьте $y_i = w_0 + w_1 x_i + \varepsilon_i$ в формулу $\hat{w}_1$ и используйте $\mathbb{E}[\varepsilon_i]=0$.)
5. Обсудите, как гетероскедастичность влияет на свойства МНК-оценок: остаются ли они несмещёнными, как меняются их стандартные ошибки? Опишите, как работают робастные стандартные ошибки Уайта (на уровне идеи).
6. Геометрический смысл: покажите, что $\hat{Y} = X (X^T X)^{-1} X^T Y$ задаёт матрицу ортогонального проектирования $H = X (X^T X)^{-1} X^T$. Какими свойствами обладает $H$ (идемпотентность, симметричность)? Как с помощью $H$ выразить вектор остатков?




# Глава 2. Простая линейная регрессия (часть II)

## 9. Градиентный спуск — численный метод оптимизации

Аналитические формулы, выведенные в части I, дают точное решение для коэффициентов $w_0$ и $w_1$ при одном признаке. Однако в машинном обучении часто встречаются задачи с десятками, сотнями и тысячами признаков. В таких случаях вычисление обратной матрицы $(X^T X)^{-1}$ становится слишком затратным или даже невозможным из‑за плохой обусловленности. Тогда на помощь приходит итеративный метод — **градиентный спуск**.

Представьте туманный горный ландшафт. Вы стоите на склоне и хотите добраться до самой нижней точки долины. Вы не видите всей картины, но можете определить направление, в котором склон уходит вниз круче всего. Делая маленький шаг в этом направлении, вы оказываетесь чуть ниже, затем снова оцениваете уклон и повторяете процедуру. Рано или поздно вы придёте к минимуму. В роли ландшафта выступает функция потерь $J(w_0, w_1) = \sum (y_i - w_0 - w_1 x_i)^2$, а направление самого крутого **возрастания** указывает градиент $\nabla J$. Чтобы спускаться, нужно двигаться в противоположном направлении — против градиента.

Правило обновления на каждом шаге:

$$
w_0^{(\text{нов})} = w_0^{(\text{стар})} - \alpha \frac{\partial J}{\partial w_0},\qquad
w_1^{(\text{нов})} = w_1^{(\text{стар})} - \alpha \frac{\partial J}{\partial w_1},
$$

где $\alpha > 0$ — **скорость обучения** (learning rate). Слишком маленький $\alpha$ замедляет сходимость, слишком большой — может вызвать расходимость или «перепрыгивание» через минимум.

Вспомним частные производные из раздела 4 части I:

$$
\frac{\partial J}{\partial w_0} = -2 \sum_{i=1}^{n} (y_i - \hat{y}_i),\qquad
\frac{\partial J}{\partial w_1} = -2 \sum_{i=1}^{n} x_i (y_i - \hat{y}_i),
$$

где $\hat{y}_i = w_0 + w_1 x_i$ — текущие предсказания модели. Подстановка даёт явные формулы **пакетного градиентного спуска** (batch gradient descent):

$$
w_0 := w_0 + 2\alpha \sum_{i=1}^{n} (y_i - \hat{y}_i),\qquad
w_1 := w_1 + 2\alpha \sum_{i=1}^{n} x_i (y_i - \hat{y}_i).
$$

На каждой итерации мы просчитываем прогнозы для всей обучающей выборки и затем сдвигаем веса. Для очень больших $n$ существуют стохастический и мини‑пакетный варианты, но идея остаётся той же.

Зачем изучать градиентный спуск в простой регрессии, где есть точное решение? Исключительно ради педагогической ценности: метод осваивается на простом примере, без отвлечения на многомерные тонкости. В последующих главах, когда число признаков вырастет, вы будете воспринимать градиентный спуск как естественное продолжение уже понятой идеи.

**Единый блок кода для всех визуализаций** (загрузка данных, масштабирование, функция потерь, 3D‑ландшафт, кривые обучения, интерактивные графики):



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# 1. Загрузка и подготовка данных
# ------------------------------------------------------------
dataset = pd.read_csv(
    "https://edunet.kea.su/repo/EduNet-web_dependencies/datasets/student_scores.csv"
)
x = dataset.iloc[:, :-1].values  # часы
y = dataset.iloc[:, 1].values    # баллы

x_train, _, y_train, _ = train_test_split(x, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_train).flatten()
y_np = y_train

# ------------------------------------------------------------
# 2. Функция потерь (MSE)
# ------------------------------------------------------------
def mse(w0, w1):
    preds = w0 + w1 * x_scaled
    return np.mean((y_np - preds) ** 2)

# ------------------------------------------------------------
# 3. Сетка для 3D-поверхности
# ------------------------------------------------------------
w0_range = np.linspace(40, 70, 80)
w1_range = np.linspace(10, 30, 80)
W0, W1 = np.meshgrid(w0_range, w1_range)
MSE_grid = np.zeros_like(W0)
for i in range(len(w0_range)):
    for j in range(len(w1_range)):
        MSE_grid[j, i] = mse(w0_range[i], w1_range[j])

# ------------------------------------------------------------
# 4. Градиентный спуск для 3D-траектории (α=0.1)
# ------------------------------------------------------------
w = np.array([50.0, 15.0])
alpha_3d = 0.1
steps_3d = 50
path_3d = [w.copy()]
for _ in range(steps_3d):
    preds = w[0] + w[1] * x_scaled
    error = preds - y_np
    grad = 2 * np.array([np.mean(error), np.mean(error * x_scaled)])
    w = w - alpha_3d * grad
    path_3d.append(w.copy())
path_3d = np.array(path_3d)

# ------------------------------------------------------------
# 5. Истинные МНК-коэффициенты (для ориентиров)
# ------------------------------------------------------------
X_design = np.vstack([np.ones_like(x_scaled), x_scaled]).T
w_ols = np.linalg.inv(X_design.T @ X_design) @ X_design.T @ y_np
w0_true, w1_true = w_ols[0], w_ols[1]



### Визуализация ландшафта функции потерь

Чтобы лучше понять, как работает градиентный спуск, построим **трёхмерный ландшафт** среднеквадратичной ошибки $J(w_0, w_1)$ для наших данных (часы подготовки → баллы).  
График показывает, что поверхность имеет форму **выпуклой чаши** с единственным глобальным минимумом.  
На него же нанесена траектория градиентного спуска — цепочка шагов, ведущая от стартовой точки к «дну».

Вращая график мышью, можно увидеть, как направление спуска всегда перпендикулярно линиям уровня, а шаги становятся короче по мере приближения к минимуму.



In [ ]:
fig = go.Figure()

fig.add_trace(go.Surface(
    x=w0_range, y=w1_range, z=MSE_grid,
    colorscale='RdYlGn_r', opacity=0.7,
    contours={"z": {"show": True, "usecolormap": True,
                     "highlightcolor": "black", "project": {"z": True}}},
    name='MSE surface'
))

fig.add_trace(go.Scatter3d(
    x=path_3d[:,0], y=path_3d[:,1],
    z=np.array([mse(p[0], p[1]) for p in path_3d]),
    mode='lines+markers',
    marker=dict(size=3, color='black'),
    line=dict(color='black', width=4),
    name='Градиентный спуск'
))

fig.add_trace(go.Scatter3d(
    x=[path_3d[0,0]], y=[path_3d[0,1]], z=[mse(path_3d[0,0], path_3d[0,1])],
    mode='markers', marker=dict(size=6, color='blue'), name='Старт'
))

fig.add_trace(go.Scatter3d(
    x=[path_3d[-1,0]], y=[path_3d[-1,1]], z=[mse(path_3d[-1,0], path_3d[-1,1])],
    mode='markers', marker=dict(size=6, color='red'), name='Финиш'
))

fig.update_layout(
    title="3D-ландшафт MSE и траектория градиентного спуска",
    scene=dict(xaxis_title="w0 (intercept)", yaxis_title="w1 (slope)", zaxis_title="MSE"),
    width=900, height=700
)
fig.show()


На графике хорошо видно:

- **Выпуклость** MSE — любой локальный минимум является глобальным, поэтому градиентный спуск не застревает.
- **Направление спуска** — вектор градиента (перпендикулярный линиям уровня) указывает кратчайший путь к минимуму.
- **Уменьшение шагов** — вблизи дна градиент становится меньше, и алгоритм автоматически замедляется.

Такой ландшафт становится «овражным» (сильно вытянутым), если признаки не масштабированы, из‑за чего спуск резко замедляется — именно поэтому мы применяем `StandardScaler` (подробнее обсуждается в разделе «Необходимость нормализации»).

### Кривые обучения при разных $\alpha$

Трёхмерный ландшафт показывает *куда* мы движемся. Не менее важно понимать, *с какой скоростью* мы сходимся к минимуму. Построим **кривые обучения** — зависимость MSE от номера итерации — для трёх характерных значений $\alpha$.



In [ ]:
def run_gd(alpha, steps=30):
    w = np.array([50.0, 15.0])
    losses = []
    for _ in range(steps):
        preds = w[0] + w[1] * x_scaled
        error = preds - y_np
        grad = 2 * np.array([np.mean(error), np.mean(error * x_scaled)])
        w = w - alpha * grad
        losses.append(mse(w[0], w[1]))
    return losses

alphas_to_test = [0.001, 0.1, 0.5]
plt.figure(figsize=(8, 5))
for a in alphas_to_test:
    losses = run_gd(a, steps=30)
    plt.plot(losses, label=f'α = {a}')
plt.xlabel('Итерация')
plt.ylabel('MSE')
plt.title('Сходимость градиентного спуска при разных α')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


**Интерпретация:**
- **α = 0.001** — слишком маленький шаг. MSE почти не уменьшается, алгоритм «ползёт».
- **α = 0.1** — хорошая скорость: ошибка быстро и плавно падает к минимуму.
- **α = 0.5** — слишком большой шаг: ошибка «взрывается», алгоритм расходится.

### Интерактивный график: влияние $\alpha$ на градиентный спуск

Теперь дадим возможность в реальном времени менять $\alpha$ и наблюдать, как это влияет на прямую регрессии и траекторию весов.



In [ ]:
# Подготовка данных для анимации
alphas = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
frames = []
x_line = np.linspace(x_scaled.min(), x_scaled.max(), 100)

def gradient_descent_full(alpha, steps=30):
    w = np.array([50.0, 15.0])
    hist = {'w0': [w[0]], 'w1': [w[1]], 'mse': [mse(w[0], w[1])]}
    for _ in range(steps):
        preds = w[0] + w[1] * x_scaled
        error = preds - y_np
        grad = 2 * np.array([np.mean(error), np.mean(error * x_scaled)])
        w = w - alpha * grad
        hist['w0'].append(w[0])
        hist['w1'].append(w[1])
        hist['mse'].append(mse(w[0], w[1]))
    return hist

for a in alphas:
    hist = gradient_descent_full(a, steps=30)
    w0_final, w1_final = hist['w0'][-1], hist['w1'][-1]

    frame_data = [
        go.Scatter(x=x_scaled, y=y_np, mode='markers',
                   marker=dict(color='blue', size=8, opacity=0.6), name='Данные'),
        go.Scatter(x=x_line, y=w0_final + w1_final * x_line, mode='lines',
                   line=dict(color='red', width=3), name=f'Прямая'),
        go.Scatter(x=hist['w0'], y=hist['w1'], mode='lines+markers',
                   marker=dict(size=4, color='green'),
                   line=dict(color='green', width=2), name='Путь весов'),
        go.Scatter(x=[hist['w0'][0]], y=[hist['w1'][0]],
                   mode='markers', marker=dict(color='blue', size=12, symbol='circle'),
                   name='Старт'),
        go.Scatter(x=[hist['w0'][-1]], y=[hist['w1'][-1]],
                   mode='markers', marker=dict(color='red', size=12, symbol='star'),
                   name='Финиш'),
        go.Scatter(x=[w0_true], y=[w1_true],
                   mode='markers', marker=dict(color='gold', size=14, symbol='diamond'),
                   name='МНК-оптимум')
    ]
    frames.append(go.Frame(data=frame_data, name=str(a)))

# Начальное состояние (α=0.1)
init_hist = gradient_descent_full(0.1, steps=30)
init_data = [
    go.Scatter(x=x_scaled, y=y_np, mode='markers',
               marker=dict(color='blue', size=8, opacity=0.6), name='Данные'),
    go.Scatter(x=x_line, y=init_hist['w0'][-1] + init_hist['w1'][-1] * x_line,
               mode='lines', line=dict(color='red', width=3), name='Прямая'),
    go.Scatter(x=init_hist['w0'], y=init_hist['w1'], mode='lines+markers',
               marker=dict(size=4, color='green'),
               line=dict(color='green', width=2), name='Путь весов'),
    go.Scatter(x=[init_hist['w0'][0]], y=[init_hist['w1'][0]],
               mode='markers', marker=dict(color='blue', size=12, symbol='circle'),
               name='Старт'),
    go.Scatter(x=[init_hist['w0'][-1]], y=[init_hist['w1'][-1]],
               mode='markers', marker=dict(color='red', size=12, symbol='star'),
               name='Финиш'),
    go.Scatter(x=[w0_true], y=[w1_true],
               mode='markers', marker=dict(color='gold', size=14, symbol='diamond'),
               name='МНК-оптимум')
]

# Слайдер
steps_slider = []
for i, a in enumerate(alphas):
    steps_slider.append(dict(
        method='animate',
        args=[[str(a)], dict(mode='immediate', frame=dict(duration=500, redraw=True))],
        label=f'α = {a}'
    ))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Данные и прямая', 'Траектория весов (w0, w1)'),
    column_widths=[0.5, 0.5]
)

for trace in init_data[:2]:
    fig.add_trace(trace, row=1, col=1)
for trace in init_data[2:]:
    fig.add_trace(trace, row=1, col=2)

fig.update_xaxes(title_text="Часы (scaled)", row=1, col=1)
fig.update_yaxes(title_text="Баллы", row=1, col=1)
fig.update_xaxes(title_text="w0", row=1, col=2)
fig.update_yaxes(title_text="w1", row=1, col=2)

fig.update(frames=frames)
fig.update_layout(
    sliders=[dict(
        active=5,  # начальное α=0.1
        currentvalue={"prefix": "Скорость обучения: "},
        pad={"t": 50},
        steps=steps_slider
    )],
    title="Интерактивный градиентный спуск — влияние α",
    width=1200, height=500
)
fig.show()


Двигая ползунок, можно наглядно увидеть, как при малых $\alpha$ прямая почти не меняется, а при слишком больших — «улетает» далеко от данных. Оптимальное значение приводит к быстрой сходимости к МНК‑решению.

### Ручной подбор параметра $w_1$

Чтобы лучше понять связь между параметрами и ошибкой, построим интерактивный график, где мы вручную меняем наклон $w_1$, а $w_0$ зафиксирован на оптимальном значении. На правом графике отображается кривая MSE$(w_1)$, и мы видим, как текущая ошибка соотносится с минимумом.



In [ ]:
w0_fixed = w0_true
w1_values = np.arange(5, 35, 0.5)
mse_values = np.array([mse(w0_fixed, w1) for w1 in w1_values])
optimal_idx = np.argmin(mse_values)
w1_optimal = w1_values[optimal_idx]

frames_w1 = []
for w1 in w1_values:
    y_line = w0_fixed + w1 * x_line
    current_mse = mse(w0_fixed, w1)
    frame_data = [
        go.Scatter(x=x_scaled, y=y_np, mode='markers',
                   marker=dict(color='blue', size=8, opacity=0.6), name='Данные'),
        go.Scatter(x=x_line, y=y_line, mode='lines',
                   line=dict(color='red', width=3), name=f'w1={w1:.1f}'),
        go.Scatter(x=w1_values, y=mse_values, mode='lines',
                   line=dict(color='green', width=2), name='MSE(w1)'),
        go.Scatter(x=[w1], y=[current_mse], mode='markers',
                   marker=dict(color='red', size=12, symbol='circle'),
                   name=f'MSE={current_mse:.1f}'),
        go.Scatter(x=[w1_optimal], y=[mse_values[optimal_idx]],
                   mode='markers', marker=dict(color='gold', size=12, symbol='star'),
                   name=f'Оптимум w1*={w1_optimal:.2f}')
    ]
    frames_w1.append(go.Frame(data=frame_data, name=str(w1)))

init_w1 = 15.0
init_mse = mse(w0_fixed, init_w1)
init_data_w1 = [
    go.Scatter(x=x_scaled, y=y_np, mode='markers',
               marker=dict(color='blue', size=8, opacity=0.6), name='Данные'),
    go.Scatter(x=x_line, y=w0_fixed + init_w1 * x_line, mode='lines',
               line=dict(color='red', width=3), name=f'w1={init_w1:.1f}'),
    go.Scatter(x=w1_values, y=mse_values, mode='lines',
               line=dict(color='green', width=2), name='MSE(w1)'),
    go.Scatter(x=[init_w1], y=[init_mse], mode='markers',
               marker=dict(color='red', size=12, symbol='circle'),
               name=f'MSE={init_mse:.1f}'),
    go.Scatter(x=[w1_optimal], y=[mse_values[optimal_idx]],
               mode='markers', marker=dict(color='gold', size=12, symbol='star'),
               name=f'Оптимум w1*={w1_optimal:.2f}')
]

steps_w1 = []
for i, w1 in enumerate(w1_values):
    steps_w1.append(dict(
        method='animate',
        args=[[str(w1)], dict(mode='immediate', frame=dict(duration=100, redraw=True))],
        label=f'w1 = {w1:.1f}'
    ))

fig_w1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Данные и прямая', 'MSE как функция w1'),
    column_widths=[0.5, 0.5]
)

for trace in init_data_w1[:2]:
    fig_w1.add_trace(trace, row=1, col=1)
for trace in init_data_w1[2:]:
    fig_w1.add_trace(trace, row=1, col=2)

fig_w1.update_xaxes(title_text="Часы (scaled)", row=1, col=1)
fig_w1.update_yaxes(title_text="Баллы", row=1, col=1)
fig_w1.update_xaxes(title_text="w1 (slope)", row=1, col=2)
fig_w1.update_yaxes(title_text="MSE", row=1, col=2)

init_idx = np.argmin(np.abs(w1_values - init_w1))
fig_w1.update(frames=frames_w1)
fig_w1.update_layout(
    sliders=[dict(
        active=init_idx,
        currentvalue={"prefix": "w1 = "},
        pad={"t": 50},
        steps=steps_w1
    )],
    title="Ручной подбор параметра w1 (w0 фиксирован на МНК-значении)",
    width=1200, height=500
)
fig_w1.show()


Здесь мы явно видим, что минимум MSE достигается в точке, где производная обращается в ноль — это и есть оптимальный наклон, найденный МНК.

> **Углублённый взгляд: сходимость и выбор $\alpha$**  
> Поскольку функция потерь выпукла и гладка, градиентный спуск гарантированно сходится к глобальному минимуму при достаточно малом $\alpha$. На практике $\alpha$ подбирают эмпирически, часто наблюдая за убыванием функции потерь. Слишком большой шаг приводит к осцилляциям или расходимости. На практике функцию потерь нередко определяют как $\frac{1}{2n}SSE$, чтобы в градиентах исчез множитель 2 и формулы стали чуть чище. Здесь мы сохраняем прямую связь с SSE. Существуют методы автоматической адаптации шага (AdaGrad, Adam и др.), но для понимания основ достаточно фиксированного $\alpha$.



## 10. Метрики качества: SSE, MSE, RMSE и R²

В части I мы ввели SSE и MSE, а также коэффициент детерминации $R^2$. Теперь дополним картину метрикой RMSE и обсудим, когда какую из них применять.

**RMSE (Root Mean Squared Error)** возвращает ошибку к исходным единицам измерения. MSE выражается в квадратных единицах (например, «тысячи долларов в квадрате»), что неудобно для интерпретации. Извлечение квадратного корня решает проблему:

$$
RMSE = \sqrt{MSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}.
$$

Теперь можно сказать: «в среднем модель ошибается на столько‑то тысяч долларов». Это самая интуитивно понятная метрика среди трёх.

**Когда что использовать:**

| Метрика | Формула | Применение |
|---------|---------|-------------|
| **SSE** | $\sum (y_i - \hat{y}_i)^2$ | Вывод формул, оптимизация. Растёт с $n$. |
| **MSE** | $SSE/n$ | Сравнение моделей на выборках разного размера. Смещённая, но состоятельная оценка $\sigma^2$. |
| **RMSE** | $\sqrt{MSE}$ | Презентация результатов, общение с бизнесом. |
| **$R^2$** | $1 - SSE/SST$ | Доля объяснённой дисперсии, быстрая оценка силы связи. |

**Важное уточнение про несмещённость.** Из части I: MLE-оценка $\hat{\sigma}^2_{\text{MLE}} = MSE$ смещена. Несмещённая оценка дисперсии ошибок: $s^2 = \frac{SSE}{n-2}$. MSE состоятельна: при $n \to \infty$ сходится к $\sigma^2$. На малых выборках лучше использовать $s^2$.

Все перечисленные метрики оценивают модель на тех же данных, на которых она обучена. Чтобы получить представление о предсказательной силе на новых наблюдениях, данные обычно разделяют на обучающую и тестовую выборки — эта практика будет обсуждаться в следующих главах.

> **Углублённый взгляд: MAE и другие метрики**  
> Кроме RMSE, существует средняя абсолютная ошибка $MAE = \frac{1}{n}\sum |y_i - \hat{y}_i|$. Она менее чувствительна к большим выбросам, чем RMSE (где выбросы из‑за возведения в квадрат оказывают непропорционально сильное влияние). Выбор метрики зависит от целей: если крупные ошибки особенно нежелательны, RMSE предпочтительнее; если все ошибки одинаково важны, MAE может быть уместнее.





## 11. Диагностика модели: анализ остатков — полное руководство с примером на Python

Построив модель и оценив её качество с помощью числовых метрик $(R^2$, RMSE), нельзя останавливаться. Числовые показатели могут быть обманчивы, если нарушены предположения, на которых строилась модель. Основной инструмент проверки этих предположений — **анализ остатков**.

Остатками называются величины  
$$
e_i = y_i - \hat{y}_i = y_i - (w_0 + w_1 x_i), \qquad i = 1,\dots,n.
$$  
Они являются наблюдаемой реализацией ненаблюдаемых случайных ошибок $varepsilon_i$ в истинной модели $y_i = w_0 + w_1 x_i + \varepsilon_i$. Если модель специфицирована верно и выполняются классические предположения (линейность, независимость ошибок, гомоскедастичность, нормальность), то остатки должны вести себя как белый шум:  
- их среднее должно быть близко к нулю $$mathbb{E}[e_i] \approx 0$);  
- дисперсия постоянна $$mathrm{Var}(e_i) \approx \text{const}$);  
- отсутствует корреляция между разными остатками;  
- распределение близко к нормальному (для малых выборок).  

Любое отклонение от этой идеальной картины указывает на конкретное нарушение и подсказывает путь улучшения модели.

**Сквозной пример.**  
Для демонстрации диагностических методов создадим искусственный набор данных, который имитирует реальную задачу: предсказание цены квартиры по её площади. Намеренно введём два типичных нарушения:  
- **нелинейность** (квадратичный член);  
- **гетероскедастичность** (дисперсия ошибки растёт с площадью).

Все вычисления будем проводить в среде Python. Сначала выполните импорты:



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.graphics.gofplots import qqplot
from scipy.stats import shapiro, chi2
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.nonparametric.smoothers_lowess import lowess

**Генерация данных.**  
Для воспроизводимости результатов зафиксируем генератор случайных чисел:



In [ ]:
np.random.seed(42)
n = 40
x = np.random.uniform(30, 150, n)
x



Истинная зависимость имеет вид  
$$
y_i = 20 + 3.5\,x_i + 0.005\,(x_i-90)^2 + 5\sqrt{x_i/80}\,\cdot\,\varepsilon_i, \quad \varepsilon_i \sim \mathcal{N}(0,1).
$$  
Параметр $5$ задаёт базовую волатильность ошибки (среднеквадратическое отклонение около 5 при $x=80$). Множитель $sqrt{x_i/80}$ вводит гетероскедастичность: дисперсия ошибки пропорциональна площади. При $x=80$ дисперсия равна $25$, при $x=30$ — около $25 \cdot 30/80 \approx 9.4$, при $x=150$ — $25 \cdot 150/80 \approx 46.9$. Реализуем это в коде:



In [ ]:
true_w0, true_w1 = 20, 3.5
nonlin = 0.005 * (x - 90)**2
noise = np.random.normal(0, 1, n) * 5 * np.sqrt(x/80)
y = true_w0 + true_w1 * x + nonlin + noise

#Визуализация данных.
#Постройте график точек (площадь, цена):
plt.figure(figsize=(8,5))
plt.scatter(x, y)
plt.xlabel('Площадь, м²')
plt.ylabel('Цена, тыс. долл.')
plt.title('Данные о квартирах')
plt.show()



На полученном графике вы увидите положительную корреляцию, а также увеличивающийся разброс точек с ростом площади — это визуальный признак гетероскедастичности.

**Подгонка линейной модели МНК.**  
Добавим к вектору $x$ столбец единиц и обучим модель:



In [ ]:
X = sm.add_constant(x)
model = sm.OLS(y, X).fit()
print(model.summary())



Метод `summary()` выдаст таблицу с коэффициентами, стандартными ошибками, t-статистиками, p‑значениями и $R^2$. Обратите внимание на величину $R^2$ и на стандартные ошибки — они могут быть занижены из-за гетероскедастичности, поэтому необходима диагностика остатков.

**Извлечение остатков и предсказанных значений:**



In [ ]:
fitted = model.fittedvalues
residuals = model.resid



**11.1. График остатков против предсказанных значений (Residuals vs Fitted)**  
Этот график является центральным диагностическим инструментом. По оси абсцисс откладываются $hat{y}_i$, по оси ординат — $e_i$. При выполнении всех предположений точки должны быть случайно рассеяны вокруг горизонтальной линии $e=0$ без какого-либо тренда или систематического изменения разброса. Добавим сглаживающую линию LOWESS (Locally Weighted Scatterplot Smoothing), которая помогает выявить нелинейные тренды.



In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(fitted, residuals, alpha=0.7)
plt.axhline(0, color='red', linestyle='--', linewidth=1.5)
lowess_line = lowess(residuals, fitted, frac=0.6)
plt.plot(lowess_line[:,0], lowess_line[:,1], color='green', linewidth=2)
plt.xlabel('Предсказанные значения (тыс. долл.)')
plt.ylabel('Остатки (тыс. долл.)')
plt.title('Residuals vs Fitted')
plt.show()


**Что вы увидите:**  
- **Веерообразную форму** — при малых предсказанных значениях разброс остатков невелик, при больших — значительно шире. Это классический признак **гетероскедастичности** (непостоянства дисперсии).  
- Зелёная сглаженная линия не будет строго горизонтальной — она может иметь лёгкий изгиб (U‑образность), что указывает на возможную **нелинейность** связи (в нашей модели пропущен квадратичный член).  

Таким образом, один график позволяет одновременно заподозрить два нарушения.

**11.2. График остатков против независимой переменной $x$**  
Для простой линейной регрессии $hat{y}_i$ линейно зависит от $x_i$, поэтому график «остатки против $x$» будет похож на предыдущий, однако он полезен для привязки к исходному масштабу признака.






In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(x, residuals, alpha=0.7)
plt.axhline(0, color='red', linestyle='--', linewidth=1.5)
plt.xlabel('Площадь (м²)')
plt.ylabel('Остатки (тыс. долл.)')
plt.title('Residuals vs x')
plt.show()


На этом графике также виден рост разброса с площадью и слабая U‑образность.

**11.3. Q‑Q plot (квантиль-квантильный график) для проверки нормальности**  
Q‑Q plot сравнивает выборочные квантили остатков с теоретическими квантилями нормального распределения. Если остатки распределены нормально, точки должны лежать на прямой линии (в идеале — на биссектрисе).


In [ ]:
fig = qqplot(residuals, line='s')
plt.title('Q-Q plot остатков')
plt.show()


**Интерпретация:**  
Если точки в основном следуют прямой, а отклонения в хвостах невелики, то гипотеза о нормальности не отвергается. Для выборки $n=40$ небольшие отклонения допустимы.

**11.4. Scale‑Location график**  
Этот график предназначен специально для обнаружения гетероскедастичности. Вместо обычных остатков $e_i$ используют **стьюдентизированные остатки**  
$$
r_i = \frac{e_i}{s_{(i)}\sqrt{1-h_{ii}}},
$$  
где $s_{(i)}$ — оценка среднеквадратического отклонения ошибки, полученная без $i$-го наблюдения, а $h_{ii}$ — диагональные элементы «шляпной» матрицы $H = X(X^T X)^{-1} X^T$ (так называемые «рычаги»). Стьюдентизированные остатки имеют одинаковую дисперсию независимо от положения наблюдения. По вертикали откладывают $sqrt{|r_i|}$ (чтобы смягчить влияние больших выбросов), по горизонтали — предсказанные значения $hat{y}_i$. Добавляется сглаживающая линия LOWESS. При гомоскедастичности эта линия должна быть горизонтальной.



In [ ]:
influence = model.get_influence()
stud_res = influence.resid_studentized_internal
sqrt_abs_resid = np.sqrt(np.abs(stud_res))

plt.figure(figsize=(8,6))
plt.scatter(fitted, sqrt_abs_resid, alpha=0.7)
lowess_line = lowess(sqrt_abs_resid, fitted, frac=0.6)
plt.plot(lowess_line[:,0], lowess_line[:,1], color='red', linewidth=2)
plt.xlabel('Предсказанные значения (тыс. долл.)')
plt.ylabel('√|Стьюдентизированные остатки|')
plt.title('Scale-Location')
plt.show()


**Что вы увидите:**  
Красная сглаженная линия будет возрастать — это надёжное подтверждение гетероскедастичности.

> **Углублённый взгляд: стандартизованные и стьюдентизированные остатки**  
> Обычные остатки $e_i$ имеют дисперсию $mathrm{Var}(e_i) = \sigma^2(1 - h_{ii})$, которая зависит от «рычага» $h_{ii}$. Поэтому для диагностики удобнее использовать **стандартизованные остатки** $frac{e_i}{\sqrt{s^2(1-h_{ii})}}$ или **стьюдентизированные**, где вместо $s^2$ берётся $s_{(i)}^2$ (оценка, не зависящая от $i$-го наблюдения). Стьюдентизированные остатки более чувствительны к выбросам и имеют t-распределение с $n-p-1$ степенями свободы.

**Вывод по визуальной диагностике:**  
В сгенерированных данных присутствуют два нарушения: (1) непостоянство дисперсии (гетероскедастичность) и (2) возможная нелинейность. Для количественной проверки перейдём к формальным статистическим тестам.

## 12. Гомоскедастичность и гетероскедастичность — теория, тесты и коррекция

**12.1. Определения и последствия**  
**Гомоскедастичность** (однородность дисперсии) — предположение, что дисперсия случайной ошибки одинакова для всех наблюдений: $mathrm{Var}$varepsilon_i) = \sigma^2 = const$.  
**Гетероскедастичность** — нарушение этого условия: $mathrm{Var}$varepsilon_i) = \sigma_i^2$, где $sigma_i^2$ зависит от $i$ (например, от $x_i$).

**Последствия гетероскедастичности для МНК:**  
- Оценки коэффициентов $hat{w}_0, \hat{w}_1$ остаются **несмещёнными** и **состоятельными** (при выполнении остальных условий).  
- Однако **стандартные ошибки** коэффициентов, вычисленные по классической формуле  
  $$
  \widehat{\mathrm{SE}}(w_1) = \frac{s}{\sqrt{\sum (x_i-\bar{x})^2}}, \quad s^2 = \frac{SSE}{n-2},
  $$  
  становятся **смещёнными** (чаще всего заниженными). Это приводит к завышению t-статистик, сужению доверительных интервалов и, как следствие, к ошибочному отвержению нулевой гипотезы о незначимости коэффициентов.

**12.2. Тест Бройша–Пагана (Breusch–Pagan)**  
Идея: оценить вспомогательную регрессию квадратов остатков на исходные регрессоры (в простом случае — на $x_i$):  
$$
e_i^2 = \alpha_0 + \alpha_1 x_i + u_i.
$$  
Если коэффициент $alpha_1$ значимо отличается от нуля, то дисперсия ошибок зависит от $x$, что говорит о гетероскедастичности. Нулевая гипотеза: $alpha_1 = 0$ (гомоскедастичность). Статистика теста $LM = n \cdot R^2_{\text{aux}}$ асимптотически распределена как $chi^2(1)$.

В Python это реализуется функцией `het_breuschpagan`:



In [ ]:
bp_test = het_breuschpagan(residuals, X)
labels = ['LM Statistic', 'LM-Test p-value', 'F-Statistic', 'F-Test p-value']
bp_results = dict(zip(labels, bp_test))
print(bp_results)



Обратите внимание на p‑значение. Если оно меньше 0.05, нулевая гипотеза отвергается — гетероскедастичность статистически значима.

**12.3. Тест Уайта (White)**  
Тест Уайта более общий, так как не предполагает конкретной функциональной формы зависимости дисперсии от регрессоров. Вспомогательная регрессия для простой модели включает $x_i$ и $x_i^2$:  
$$
e_i^2 = \alpha_0 + \alpha_1 x_i + \alpha_2 x_i^2 + u_i.
$$  
Нулевая гипотеза: $alpha_1 = \alpha_2 = 0$. Статистика $LM = n \cdot R^2_{\text{aux}}$ имеет асимптотическое распределение $chi^2(2)$.

Реализация:


In [ ]:
X_white = sm.add_constant(np.column_stack((x, x**2)))
white_model = sm.OLS(residuals**2, X_white).fit()
white_lm = len(y) * white_model.rsquared
df_white = white_model.df_model
p_white = 1 - chi2.cdf(white_lm, df_white)
print(f"White test LM statistic: {white_lm:.3f}, p-value: {p_white:.4f}")


Если p‑value < 0.05, гипотеза о гомоскедастичности отвергается. Оба теста обычно дают согласованные результаты.

**12.4. Стратегии коррекции гетероскедастичности**  
В зависимости от целей и природы данных можно использовать один из трёх основных подходов.

**1. Робастные стандартные ошибки (Huber–White).**  
Этот метод не изменяет сами коэффициенты, но корректирует их стандартные ошибки и ковариационную матрицу. Оценка ковариационной матрицы имеет «сэндвичную» форму:  
$$
\widehat{\mathrm{Cov}}\hat{w}) = (X^T X)^{-1} (X^T \Omega X) (X^T X)^{-1},
$$  
где $Omega = \mathrm{diag}(e_1^2, \dots, e_n^2)$. Различные варианты (HC0, HC1, HC2, HC3) отличаются поправками на степени свободы и рычаги. В `statsmodels` используется `cov_type='HC1'`:



In [ ]:
robust_model = model.get_robustcov_results(cov_type='HC1')
print(robust_model.summary())


Сравните стандартные ошибки в полученной таблице с классическими. Робастные ошибки будут больше, доверительные интервалы шире, что отражает истинную неопределённость при гетероскедастичности.

**2. Преобразование переменных (логарифмирование).**  
Если дисперсия ошибки пропорциональна квадрату среднего, то логарифмирование зависимой и независимой переменных часто стабилизирует дисперсию. Перейдём к log‑log модели:



In [ ]:
y_log = np.log(y)
x_log = np.log(x)
X_log = sm.add_constant(x_log)
model_log = sm.OLS(y_log, X_log).fit()
print(model_log.summary())



После этого повторно построим график остатков:



In [ ]:
fitted_log = model_log.fittedvalues
resid_log = model_log.resid
plt.scatter(fitted_log, resid_log)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Предсказанные логарифмы цен')
plt.ylabel('Остатки')
plt.title('Residuals vs Fitted (log-log модель)')
plt.show()





Картина станет равномернее, веер исчезнет. Повторите тест Бройша–Пагана на новой модели — p‑значение станет больше 0.05. **Интерпретация коэффициентов:** теперь они интерпретируются как эластичности — увеличение площади на 1% приводит к увеличению цены на $w_1\%$ в среднем.

**3. Взвешенный метод наименьших квадратов (WLS).**  
Если известна форма гетероскедастичности, например $mathrm{Var}$varepsilon_i) = \sigma^2 x_i$, то оптимальными весами будут $w_i = 1/x_i$. Взвешенный МНК минимизирует $sum w_i (y_i - w_0 - w_1 x_i)^2$, что эквивалентно преобразованию переменных, делающему дисперсию постоянной.



In [ ]:
wls_model = sm.WLS(y, X, weights=1.0 / x).fit()
print(wls_model.summary())



Коэффициенты почти не изменятся, но стандартные ошибки станут корректными. При правильно выбранных весах WLS является эффективной оценкой (минимальная дисперсия среди линейных несмещённых).

> **Углублённый взгляд: выбор стратегии**  
> - Робастные стандартные ошибки — самый простой и быстрый способ, не требующий изменения модели. Рекомендуется, когда интерпретация коэффициентов в исходных единицах критична.  
> - Логарифмирование меняет интерпретацию (эластичности), но часто решает проблему и улучшает симметрию распределения остатков.  
> - WLS эффективен, если форма гетероскедастичности априори известна; на практике форму весов иногда оценивают (двухшаговый метод).

## 13. Проверка нормальности ошибок — теория и практика

Для того чтобы МНК-оценки были **BLUE** (Best Linear Unbiased Estimators), достаточно условий теоремы Гаусса–Маркова:  
1. $mathbb{E}[\varepsilon_i] = 0$;  
2. $mathrm{Var}$varepsilon_i) = \sigma^2$ (гомоскедастичность);  
3. $mathrm{Cov}$varepsilon_i, \varepsilon_j) = 0$ для $i\neq j$ (независимость).  

**Нормальность** ошибок $varepsilon_i \sim \mathcal{N}(0,\sigma^2)$ **не требуется** для этих свойств. Однако она необходима для **точных** t-тестов и F-тестов на малых выборках, а также для построения точных доверительных интервалов для коэффициентов. При больших выборках $(n > 30-50$) центральная предельная теорема обеспечивает приближённую нормальность распределения $hat{w}_j$, даже если сами ошибки ненормальны. На малых выборках нарушение нормальности может исказить p‑значения.

**Визуальная проверка:** Q‑Q plot уже построен в разделе 11.3. Если точки ложатся на прямую (допустимы небольшие отклонения в хвостах), то нормальность не отвергается.

**Тест Шапиро–Уилка** — формальный тест, обладающий высокой мощностью при малых и средних выборках. Нулевая гипотеза: данные извлечены из нормального распределения.



In [ ]:
stat, p = shapiro(residuals)
print(f"Shapiro-Wilk test statistic: {stat:.4f}, p-value: {p:.4f}")



Если p‑value > 0.05, нет оснований отвергать гипотезу о нормальности. Если p‑value < 0.05 (например, 0.01), то нормальность нарушена.

**Что делать при нарушении нормальности?**  

- **Асимптотический подход.** При $n \ge 50$ можно полагаться на асимптотическую нормальность оценок, и стандартные t-тесты будут приближённо корректны.  
- **Бутстреп (bootstrap).** Непараметрический метод, не требующий предположений о распределении ошибок. Он строит эмпирическое распределение интересующей статистики путём многократной выборки с возвращением из исходных данных. Покажем на примере получения доверительного интервала для коэффициента наклона $w_1$. Для воспроизводимости используем тот же seed, что и в начале: `np.random.seed(42)`, а внутри цикла будем использовать тот же генератор (через `np.random.choice`). Также для наглядности выведем классический доверительный интервал из модели и сравним с бутстреп-интервалом.


In [ ]:
# Классический доверительный интервал
ci_classical = model.conf_int()[1]  # для w1 (вторая строка, индекс 1)
print(f"Классический 95% ДИ для w1: [{ci_classical[0]:.3f}, {ci_classical[1]:.3f}]")

# Бутстреп-интервал
n_boot = 2000
w1_boot = np.empty(n_boot)
np.random.seed(42)   # фиксируем seed для воспроизводимости бутстрепа
for i in range(n_boot):
    idx = np.random.choice(len(y), size=len(y), replace=True)
    Xb = X[idx]
    yb = y[idx]
    model_b = sm.OLS(yb, Xb).fit()
    w1_boot[i] = model_b.params[1]

alpha = 0.05
ci_low, ci_high = np.percentile(w1_boot, [100*alpha/2, 100*(1-alpha/2)])
print(f"Бутстреп 95% ДИ для w1: [{ci_low:.3f}, {ci_high:.3f}]")


При нормальных остатках оба интервала будут близки. При нарушении нормальности бутстреп-интервал может оказаться шире или асимметричнее.  
- **Робастные регрессионные методы.** Например, `HuberRegressor` из `sklearn` или `RLM` из `statsmodels` используют M-оценки, устойчивые к тяжёлым хвостам.

> **Углублённый взгляд: связь с теоремой Гаусса–Маркова**  
> Теорема BLUE не требует нормальности. Она гарантирует, что среди всех линейных несмещённых оценок МНК имеет наименьшую дисперсию. Однако для перехода от дисперсии к доверительным интервалам и гипотезам необходимо знать распределение оценок. При нормальных ошибках это распределение точное (t-распределение). Без нормальности мы полагаемся на асимптотику или бутстреп.




## 14. Заключение и резюме главы

Мы проделали путь от интуитивной идеи «проведём наилучшую прямую» до полноценного статистического инструмента с развитой системой диагностики. Простая линейная регрессия — это не просто две формулы, а целый набор взаимосвязанных методов: вероятностная модель, принцип максимального правдоподобия, метод наименьших квадратов, матричная запись, градиентный спуск, метрики качества и, наконец, анализ остатков для проверки предпосылок.

Ниже представлена итоговая таблица, обобщающая ключевые предположения классической линейной регрессии, способы их проверки и возможные действия при нарушениях.

| Предпосылка | Что проверяет | Метод проверки | Что делать при нарушении |
|-------------|---------------|----------------|---------------------------|
| Линейность по параметрам | Верна ли линейная форма | График остатков vs $hat{y}$, vs $x$; RESET‑тест (тест Рамсея) | Добавить полиномиальные члены, преобразовать переменные |
| Независимость ошибок | Нет ли автокорреляции | График остатков по времени/порядку; тест Дарбина–Уотсона | Использовать модели временных рядов, кластеризованные стандартные ошибки |
| Гомоскедастичность | Постоянство дисперсии | Scale‑Location график; тесты Бройша–Пагана, Уайта | Робастные стандартные ошибки, логарифмирование, взвешенный МНК |
| Нормальность ошибок | Можно ли применять t‑ и F‑тесты | Q‑Q plot, тест Шапиро–Уилка | Асимптотические тесты (при больших $n$), бутстреп, непараметрические методы |
| Отсутствие влиятельных наблюдений | Нет ли точек, сильно меняющих оценки | Расстояние Кука, леверидж | Анализ выбросов; возможно, удаление или робастная регрессия |

Теперь вы владеете полным инструментарием простой линейной регрессии. Это надёжный фундамент для перехода к множественной регрессии, регуляризации и другим более сложным методам.

---

### Вопросы для самопроверки (часть II)

**Для начинающих:**
1. Опишите идею градиентного спуска своими словами. Чем пакетный градиентный спуск отличается от стохастического?
2. Зачем нужна метрика RMSE, если уже есть MSE? В каких единицах она измеряется?
3. Какой график остатков помогает обнаружить гетероскедастичность? Что именно на нём вы бы искали?
4. Почему нормальность ошибок не нужна для нахождения коэффициентов, но важна для проверки гипотез?
5. Какие действия можно предпринять, если обнаружена гетероскедастичность?

**Для профессионалов:**
1. Выведите формулу для стандартной ошибки $hat{w}_1$ и объясните, как она используется в t‑тесте для проверки значимости коэффициента.
2. Докажите, что МНК‑оценки $hat{w}_0$ и $hat{w}_1$ являются несмещёнными при условии $mathbb{E}[\varepsilon_i]=0$.
3. Опишите идею теста Бройша–Пагана. Почему регрессия квадратов остатков на $x$ позволяет судить о гомоскедастичности?
4. Как бутстреп помогает строить доверительные интервалы для коэффициентов без предположения о нормальности?
5. Сравните робастные стандартные ошибки Уайта и взвешенный МНК: в чём принципиальная разница подходов?

### Задания повышенной сложности

1. **Несмещённость и дисперсия $hat{w}_1$.**  
   Пусть истинная модель $y_i = w_0^* + w_1^* x_i + \varepsilon_i$, где $varepsilon_i$ независимы, $mathbb{E}[\varepsilon_i]=0$ и $text{Var}$varepsilon_i)=\sigma^2$. Используя выражение для $hat{w}_1$ из части I, покажите, что $mathbb{E}[\hat{w}_1] = w_1^*$, и выведите формулу дисперсии $text{Var}$hat{w}_1) = \sigma^2 / \sum (x_i - \bar{x})^2$.

2. **Стандартная ошибка и t‑статистика.**  
   Предположив дополнительно нормальность ошибок, постройте $100(1-\alpha)\%$ доверительный интервал для $w_1$ и проверьте гипотезу $H_0: w_1 = 0$ против $H_1: w_1 \neq 0$. Объясните, как интерпретировать результат.

Эти задания закрепят теоретические основы и подготовят вас к изучению множественной линейной регрессии, где многие из этих идей естественно обобщаются.
